# Redrob - Intelligent Candidate Ranker (Sandbox)

A runnable, end-to-end demo of the ranking system for the Senior AI Engineer JD.
Runs on CPU only, no network, no GPU, in a few seconds.

How it works: each candidate's score blends a TF-IDF semantic match over their
career *prose* (not the skills tags) with a structured rubric, then multiplies by
the JD's explicit do-NOT-want gates, a behavioral availability multiplier, a
location factor, and a honeypot sink. See the repo README for the full design.

To use: Runtime -> Run all.


## 1. The ranker (self-contained, standard library only)


In [ ]:
"""
Lightweight TF-IDF semantic model over candidate *career prose*.

Why career prose (summary + job descriptions) and NOT the skills list?
The JD is explicit: "find candidates whose skills section contains the most AI
keywords" is a trap. Real fit shows up in what a person *did* ("built a
recommendation system", "owned search relevance"), often in plain language that
never names "RAG" or "Pinecone". So our dense-ish signal is computed over the
narrative text, which is where plain-language Tier-5 fits live.

Pure-Python + numpy only (no sklearn/scipy) so the ranking step has a tiny,
fully reproducible dependency footprint and runs CPU-only with no network.
"""
import math
import re
from collections import Counter

_TOKEN_RE = re.compile(r"[a-z][a-z0-9\+\#\.\-]{1,}")

# Light stoplist - generic resume / english filler that carries no fit signal.
_STOP = set("""
a an the and or of to in on for with at by from as is are was were be been being
i we you he she it they them our your their this that these those my me us
have has had do does did will would can could should may might must not no nor
worked working work built build building using used use made make team teams
across into over under within most some many few more less very much also able
year years month months experience role roles responsible responsibility
company companies project projects help helped helping including include various
""".split())


def tokenize(text):
    if not text:
        return []
    toks = _TOKEN_RE.findall(text.lower())
    return [t for t in toks if t not in _STOP and len(t) > 1]


# ---- The JD "query": the substantive concepts the role is really about. ----
# Weights reflect the JD's own emphasis ("things you absolutely need" first).
# These are CONCEPTS, deliberately phrased the way a candidate might describe
# real work, so we reward narrative evidence over keyword tags.
JD_QUERY_WEIGHTS = {
    # Core: retrieval / ranking / search relevance (the heart of the role)
    "retrieval": 3.0, "ranking": 3.0, "rank": 2.0, "ranker": 2.5, "re-ranking": 2.5,
    "rerank": 2.5, "relevance": 2.5, "search": 2.5, "recommendation": 3.0,
    "recommender": 2.5, "recsys": 2.5, "matching": 2.0, "personalization": 2.0,
    # Embeddings / vector infra
    "embedding": 3.0, "embeddings": 3.0, "vector": 2.5, "semantic": 2.0,
    "faiss": 2.0, "pinecone": 2.0, "weaviate": 2.0, "qdrant": 2.0, "milvus": 2.0,
    "opensearch": 2.0, "elasticsearch": 2.0, "bm25": 2.5, "ann": 1.5, "knn": 1.5,
    "two-tower": 2.0, "sentence-transformers": 2.0, "bge": 1.5, "e5": 1.0,
    # Evaluation (a hard requirement in the JD)
    "ndcg": 3.0, "mrr": 2.5, "map": 1.5, "evaluation": 2.0, "eval": 1.5,
    "offline": 1.5, "online": 1.5, "experiment": 1.5, "metrics": 1.2,
    # Production / scale (JD wants shippers, not researchers)
    "production": 2.0, "deployed": 1.8, "latency": 1.8, "serving": 1.8,
    "scale": 1.5, "throughput": 1.5, "users": 1.2, "traffic": 1.5, "pipeline": 1.0,
    # ML / NLP / IR foundations
    "nlp": 2.5, "ir": 1.5, "transformer": 1.8, "transformers": 1.8, "llm": 1.8,
    "fine-tuning": 1.8, "fine-tune": 1.8, "lora": 1.5, "qlora": 1.5, "peft": 1.5,
    "learning-to-rank": 2.5, "xgboost": 1.5, "machine": 1.0, "learning": 1.0,
    "model": 1.0, "models": 1.0, "python": 1.2,
    # Light HR-tech / marketplace bonus terms
    "marketplace": 1.2, "candidate": 0.8, "recruiter": 0.8, "hiring": 0.6,
}


def build_idf(doc_freq, n_docs):
    """Smoothed idf from a {term: document_frequency} map."""
    idf = {}
    for term, df in doc_freq.items():
        idf[term] = math.log((1.0 + n_docs) / (1.0 + df)) + 1.0
    return idf


def query_vector(idf):
    """idf-weighted JD concept vector, restricted to terms seen in the corpus."""
    q = {}
    for term, w in JD_QUERY_WEIGHTS.items():
        if term in idf:
            q[term] = w * idf[term]
    norm = math.sqrt(sum(v * v for v in q.values())) or 1.0
    return q, norm


def doc_cosine(tokens, idf, qvec, qnorm):
    """Cosine similarity between a document's tf-idf vector and the JD query."""
    if not tokens:
        return 0.0
    tf = Counter(tokens)
    # sublinear tf scaling damps repetition / keyword spamming
    dot = 0.0
    sq = 0.0
    for term, c in tf.items():
        w = (1.0 + math.log(c)) * idf.get(term, 0.0)
        sq += w * w
        if term in qvec:
            dot += w * qvec[term]
    dnorm = math.sqrt(sq) or 1.0
    return dot / (dnorm * qnorm)

"""
Structured feature extraction from a candidate record.

Everything a great recruiter reads "between the lines" of the JD is turned into
explicit, inspectable features here: what the person actually DID (career prose),
whether the role is a real engineering role, whether the AI signal is trustworthy
or just keyword padding, availability, and profile *consistency* (honeypots).
"""
import datetime
import re

TODAY = datetime.date(2026, 6, 1)

STRONG_TITLE = re.compile(
    r"(machine learning|ml engineer|ml scientist|ai engineer|applied scientist|"
    r"applied ml|nlp|search engineer|recommendation|recommender|relevance|"
    r"information retrieval|data scientist|research engineer|deep learning)", re.I)
OK_TITLE = re.compile(
    r"(software engineer|backend|back-end|data engineer|analytics engineer|"
    r"full stack|platform engineer|cloud engineer|devops|sde|staff engineer|"
    r"principal engineer|developer)", re.I)
NONENG_TITLE = re.compile(
    r"(hr |human resource|recruiter|marketing|sales|account(ant|s)?|content writer|"
    r"copywriter|customer support|operations manager|project manager|program manager|"
    r"business analyst|graphic designer|ui/ux designer|civil engineer|"
    r"mechanical engineer|electrical engineer|product manager|finance|"
    r"supply chain|logistics)", re.I)

CONSULTING = {
    "tcs", "tata consultancy", "infosys", "wipro", "accenture", "cognizant",
    "capgemini", "tech mahindra", "hcl", "hcltech", "ltimindtree", "mindtree",
    "mphasis", "deloitte", "ibm services", "dxc", "genpact",
}

DOMAIN_RE = re.compile(
    r"\b(retrieval|ranking|re-?rank|relevance|recommendation|recommender|recsys|"
    r"semantic search|search relevance|vector (?:search|database|db|index)|"
    r"embedding|sentence-?transformer|bm25|elasticsearch|opensearch|faiss|"
    r"pinecone|weaviate|qdrant|milvus|learning.to.rank|ndcg|mrr|"
    r"information retrieval|nearest neighbor|two-tower|matching|personaliz)", re.I)
PROD_RE = re.compile(
    r"\b(production|deployed|deploy|real users|at scale|latency|serving|served|"
    r"throughput|requests per|qps|traffic|a/b test|ab test|rollout|shipped|"
    r"millions of|users|live)", re.I)
EVAL_RE = re.compile(
    r"\b(ndcg|mrr|\bmap\b|precision|recall|offline (?:eval|metric)|online (?:eval|metric)|"
    r"a/b test|ab test|experiment|benchmark|evaluation framework|offline-to-online)", re.I)
RESEARCH_RE = re.compile(
    r"\b(phd|ph\.d|publication|published|paper|neurips|icml|acl|emnlp|cvpr|"
    r"research lab|academic|thesis|postdoc|professor)", re.I)
CVSPEECH_RE = re.compile(
    r"\b(computer vision|image classif|object detection|segmentation|ocr|"
    r"speech recognition|asr|tts|text-to-speech|robotics|slam|lidar|"
    r"point cloud|gans?|image generation|pose estimation)", re.I)
NLP_IR_RE = re.compile(
    r"\b(nlp|natural language|retrieval|ranking|search|recommendation|"
    r"information retrieval|text|language model|embedding|bert|transformer)", re.I)
LANGCHAIN_RE = re.compile(r"\b(langchain|llamaindex|openai api|gpt-?4|prompt engineering|rag demo)", re.I)
LEAD_ONLY_RE = re.compile(r"\b(architect|tech lead|engineering manager|head of|director|vp )", re.I)
CODE_RE = re.compile(r"\b(python|java|c\+\+|golang|scala|wrote|implemented|coded|built|developed|hands-on)", re.I)

CORE_SKILLS = {
    "embeddings", "vector search", "vector databases", "faiss", "pinecone",
    "weaviate", "qdrant", "milvus", "elasticsearch", "opensearch", "bm25",
    "semantic search", "retrieval", "information retrieval", "ranking",
    "learning to rank", "recommender systems", "recommendation systems",
    "nlp", "transformers", "sentence transformers", "fine-tuning llms", "lora",
    "qlora", "peft", "pytorch", "tensorflow", "xgboost", "python", "spark",
    "rag", "llm", "ndcg", "sentence-transformers", "hugging face", "bert",
}
PROF_W = {"beginner": 0.4, "intermediate": 0.7, "advanced": 1.0, "expert": 1.15}

INDIA_T1 = re.compile(
    r"(pune|noida|hyderabad|mumbai|delhi|gurgaon|gurugram|bangalore|bengaluru|"
    r"new delhi|ncr|chennai)", re.I)


def _date(s):
    if not s:
        return None
    try:
        return datetime.date.fromisoformat(s)
    except Exception:
        return None


def career_prose(c):
    parts = []
    p = c.get("profile", {})
    parts.append(p.get("summary") or "")
    parts.append(p.get("headline") or "")
    for j in c.get("career_history", []):
        parts.append(j.get("title") or "")
        parts.append(j.get("description") or "")
    return " ".join(parts)


def honeypot_flags(c):
    """Detect 'subtly impossible' profiles (forced to tier 0 in ground truth)."""
    flags = []
    p = c.get("profile", {})
    yoe = p.get("years_of_experience") or 0
    ch = c.get("career_history", [])
    sum_months = sum((j.get("duration_months") or 0) for j in ch)
    if sum_months / 12.0 > yoe + 3.5:
        flags.append("tenure_exceeds_yoe")
    for j in ch:
        sd = _date(j.get("start_date"))
        ed = _date(j.get("end_date")) or TODAY
        dur = j.get("duration_months") or 0
        if sd:
            span = (ed.year - sd.year) * 12 + (ed.month - sd.month)
            if abs(span - dur) > 18:
                flags.append("date_duration_mismatch")
                break
    sk = c.get("skills", [])
    ghost_expert = sum(
        1 for s in sk
        if s.get("proficiency") in ("advanced", "expert")
        and (s.get("duration_months") or 0) == 0
        and (s.get("endorsements") or 0) == 0)
    if ghost_expert >= 4:
        flags.append("ghost_expert_skills")
    if ch:
        longest = max((j.get("duration_months") or 0) for j in ch)
        if longest / 12.0 > yoe + 2.5:
            flags.append("single_role_exceeds_career")
    return flags


def extract(c):
    """Return a dict of structured features used by scoring + reasoning."""
    p = c.get("profile", {})
    sig = c.get("redrob_signals", {})
    ch = c.get("career_history", [])
    title = p.get("current_title") or ""
    prose = career_prose(c)

    domain_hits = len(DOMAIN_RE.findall(prose))
    prod_hits = len(PROD_RE.findall(prose))
    eval_hits = len(EVAL_RE.findall(prose))
    research_hits = len(RESEARCH_RE.findall(prose))
    cv_hits = len(CVSPEECH_RE.findall(prose))
    nlp_hits = len(NLP_IR_RE.findall(prose))

    companies = [(j.get("company") or "").lower() for j in ch]
    industries = [(j.get("industry") or "").lower() for j in ch] + [(p.get("current_industry") or "").lower()]
    consulting_only = bool(companies) and all(
        any(k in comp for k in CONSULTING) for comp in companies)
    ever_product = any(
        ind in ("software", "fintech", "e-commerce", "food delivery", "internet",
                "ai/ml", "saas", "edtech", "adtech", "media", "healthtech ai",
                "transportation", "ai services", "insurance tech")
        for ind in industries)

    skill_trust = 0.0
    core_skill_names = []
    for s in c.get("skills", []):
        name = (s.get("name") or "").lower()
        if any(cs in name or name in cs for cs in CORE_SKILLS):
            dur = s.get("duration_months") or 0
            end = s.get("endorsements") or 0
            prof = PROF_W.get(s.get("proficiency"), 0.5)
            trust = prof * (1.0 + (dur / 12.0) ** 0.5) * (1.0 + min(end, 50) / 25.0)
            if dur == 0 and end == 0:
                trust *= 0.1
            skill_trust += trust
            if dur > 0 or end > 0:
                core_skill_names.append(s.get("name"))

    assess = sig.get("skill_assessment_scores") or {}
    assess_avg = (sum(assess.values()) / len(assess)) if assess else None

    short_stints = sum(
        1 for j in ch
        if not j.get("is_current") and 0 < (j.get("duration_months") or 0) < 18)
    n_past = sum(1 for j in ch if not j.get("is_current"))

    cur = next((j for j in ch if j.get("is_current")), None)
    cur_desc = (cur or {}).get("description", "") if cur else ""
    lead_only_current = bool(LEAD_ONLY_RE.search(title)) and not CODE_RE.search(cur_desc)

    last_active = _date(sig.get("last_active_date"))
    days_inactive = (TODAY - last_active).days if last_active else 999

    return {
        "candidate_id": c.get("candidate_id"),
        "name": p.get("anonymized_name"),
        "title": title,
        "yoe": p.get("years_of_experience") or 0,
        "company": p.get("current_company"),
        "industry": p.get("current_industry"),
        "location": p.get("location") or "",
        "country": p.get("country") or "",
        "prose": prose,
        "domain_hits": domain_hits,
        "prod_hits": prod_hits,
        "eval_hits": eval_hits,
        "research_hits": research_hits,
        "cv_hits": cv_hits,
        "nlp_hits": nlp_hits,
        "strong_title": bool(STRONG_TITLE.search(title)),
        "ok_title": bool(OK_TITLE.search(title)),
        "noneng_title": bool(NONENG_TITLE.search(title)) and not STRONG_TITLE.search(title) and not OK_TITLE.search(title),
        "consulting_only": consulting_only,
        "ever_product": ever_product,
        "skill_trust": skill_trust,
        "core_skill_names": core_skill_names[:6],
        "assess_avg": assess_avg,
        "short_stints": short_stints,
        "n_past": n_past,
        "lead_only_current": lead_only_current,
        "langchain_only": bool(LANGCHAIN_RE.search(prose)) and research_hits == 0 and domain_hits <= 1,
        "india_t1": bool(INDIA_T1.search(p.get("location") or "")),
        "in_india": (p.get("country") or "") == "India",
        "days_inactive": days_inactive,
        "honeypot": honeypot_flags(c),
        "sig": sig,
        "sig_summary": {
            "recruiter_response_rate": sig.get("recruiter_response_rate"),
            "open_to_work_flag": sig.get("open_to_work_flag"),
            "notice_period_days": sig.get("notice_period_days"),
        },
    }

"""
Scoring: turn features + semantic similarity into a single fit score.

Design = "what a great recruiter would weigh", made explicit and auditable:

  final = (0.45 * semantic + 0.55 * rubric)   # what they did + structured fit
          * disqualifier_gate                 # JD's hard 'do NOT want' rules
          * behavioral_multiplier             # are they actually reachable/available
          * location_factor                   # India Tier-1 / relocate preference
          * honeypot_factor                   # sink subtly-impossible profiles
          * experience_gate                   # soft 5-9y band preference

Each component is in [0,1] (multipliers slightly above/below 1) so the final
score stays interpretable and the pieces can be read off for the reasoning text.
"""
import math


def _clamp(x, lo, hi):
    return max(lo, min(hi, x))


def rubric_score(f):
    """Structured fit in [0,1]. Career evidence is weighted most heavily."""
    if f["strong_title"]:
        role = 1.0
    elif f["ok_title"]:
        role = 0.62
    elif f["noneng_title"]:
        role = 0.04
    else:
        role = 0.30
    if f["noneng_title"] and f["domain_hits"] >= 4 and f["prod_hits"] >= 2:
        role = 0.35

    domain = 1.0 - math.exp(-f["domain_hits"] / 4.0)
    prod = 1.0 - math.exp(-f["prod_hits"] / 3.0)
    ev = 1.0 - math.exp(-f["eval_hits"] / 2.0)
    skill = 1.0 - math.exp(-f["skill_trust"] / 6.0)

    y = f["yoe"]
    if 6 <= y <= 8:
        exp = 1.0
    elif 5 <= y < 6 or 8 < y <= 9:
        exp = 0.9
    elif 4 <= y < 5 or 9 < y <= 11:
        exp = 0.72
    elif 3 <= y < 4 or 11 < y <= 13:
        exp = 0.5
    else:
        exp = 0.3

    r = (0.30 * domain + 0.26 * role + 0.14 * prod +
         0.10 * ev + 0.10 * skill + 0.10 * exp)
    return _clamp(r, 0.0, 1.0), {
        "role": role, "domain": domain, "prod": prod,
        "ev": ev, "skill": skill, "exp": exp}


def disqualifier_gate(f):
    """Multiplicative penalties for the JD's explicit 'do NOT want' list."""
    g = 1.0
    reasons = []
    if f["research_hits"] >= 2 and f["prod_hits"] == 0:
        g *= 0.30
        reasons.append("research-leaning with no clear production signal")
    if f["consulting_only"] and not f["ever_product"]:
        g *= 0.28
        reasons.append("entire career at IT-services/consulting firms")
    if f["cv_hits"] >= 3 and f["nlp_hits"] <= 1 and f["domain_hits"] <= 1:
        g *= 0.45
        reasons.append("CV/speech/robotics focus with little NLP/IR")
    if f["langchain_only"]:
        g *= 0.55
        reasons.append("AI experience appears to be recent LLM-API wrapping only")
    if f["lead_only_current"]:
        g *= 0.62
        reasons.append("current role looks lead/architecture-only (JD wants an IC who codes)")
    if f["n_past"] >= 3 and f["short_stints"] >= 3:
        g *= 0.78
        reasons.append("frequent short stints (title-chaser pattern)")
    return g, reasons


def behavioral_multiplier(f):
    """Availability / reachability multiplier in ~[0.45, 1.12]."""
    s = f["sig"]
    m = 1.0
    notes = []
    otw = s.get("open_to_work_flag")
    if otw is True:
        m *= 1.07
    elif otw is False:
        m *= 0.82
        notes.append("not marked open-to-work")
    di = f["days_inactive"]
    if di <= 30:
        m *= 1.05
    elif di <= 90:
        m *= 1.0
    elif di <= 180:
        m *= 0.84
        notes.append("last active ~%dd ago" % di)
    else:
        m *= 0.62
        notes.append("inactive ~%dd" % di)
    rr = s.get("recruiter_response_rate")
    if rr is not None:
        if rr < 0.15:
            m *= 0.70
            notes.append("low recruiter response rate (%.0f%%)" % (rr * 100))
        elif rr < 0.35:
            m *= 0.9
        elif rr >= 0.6:
            m *= 1.05
    ic = s.get("interview_completion_rate")
    if ic is not None and ic < 0.4:
        m *= 0.9
        notes.append("low interview completion rate")
    nd = s.get("notice_period_days")
    if nd is not None:
        if nd <= 30:
            m *= 1.03
        elif nd > 90:
            m *= 0.92
            notes.append("long notice period (%dd)" % nd)
    if s.get("verified_email") and s.get("verified_phone"):
        m *= 1.01
    return _clamp(m, 0.45, 1.12), notes


def location_factor(f):
    relocate = f["sig"].get("willing_to_relocate")
    if f["india_t1"]:
        return 1.05, []
    if f["in_india"]:
        if relocate:
            return 1.0, []
        return 0.95, ["in India but outside Tier-1 hubs"]
    if relocate:
        return 0.9, ["outside India but willing to relocate"]
    return 0.78, ["outside India, relocation not indicated"]


def honeypot_factor(f):
    if f["honeypot"]:
        return 0.02, f["honeypot"]
    return 1.0, []


def experience_gate(f):
    """Soft gate (not a hard cutoff) reflecting the JD's 5-9y 'range, not a
    requirement'."""
    y = f["yoe"]
    notes = []
    if 5 <= y <= 9:
        g = 1.0
    elif 4 <= y < 5 or 9 < y <= 10:
        g = 0.95
    elif 3 <= y < 4:
        g = 0.80
        notes.append("%.1fy experience is below the 5-9y target band" % y)
    elif 10 < y <= 13:
        g = 0.85
        notes.append("%.1fy experience is above the 5-9y target band" % y)
    elif y < 3:
        g = 0.6
        notes.append("only %.1fy experience" % y)
    else:
        g = 0.7
        notes.append("%.1fy experience well above target band" % y)
    return g, notes


def score_candidate(f, semantic):
    rub, parts = rubric_score(f)
    base = 0.45 * semantic + 0.55 * rub
    gate, dq_reasons = disqualifier_gate(f)
    beh, beh_notes = behavioral_multiplier(f)
    loc, loc_notes = location_factor(f)
    hp, hp_flags = honeypot_factor(f)
    exg, exp_notes = experience_gate(f)
    final = base * gate * beh * loc * hp * exg
    detail = {
        "semantic": semantic, "rubric": rub, "rubric_parts": parts,
        "base": base, "gate": gate, "dq_reasons": dq_reasons,
        "behavioral": beh, "beh_notes": beh_notes,
        "location": loc, "loc_notes": loc_notes,
        "honeypot_factor": hp, "honeypot_flags": hp_flags,
        "exp_gate": exg, "exp_notes": exp_notes,
        "final": final,
    }
    return final, detail

"""
Generate specific, non-templated 1-2 sentence reasoning per candidate.

Stage-4 review penalizes: empty, all-identical, name-insertion templates,
hallucinated skills, and reasoning that contradicts the rank. So each string is
built ONLY from facts in the candidate's own record, leads with the single most
relevant piece of career evidence, varies its phrasing by candidate, and always
surfaces the genuine concern (if any) that affected the rank.
"""
import re

_LEADS = [
    (0.62, "Excellent fit"),
    (0.50, "Strong fit"),
    (0.40, "Good fit"),
    (0.32, "Solid candidate"),
    (0.00, "Plausible fit"),
]


def _lead(final):
    for thr, label in _LEADS:
        if final >= thr:
            return label
    return "Adjacent fit"


def _specialization(f):
    t = (f["title"] or "").lower()
    if "recommend" in t:
        return "recommendation-systems"
    if "search" in t:
        return "search/retrieval"
    if "nlp" in t:
        return "NLP & retrieval"
    if "applied scien" in t or "research engineer" in t:
        return "applied-ML"
    if f["domain_hits"] >= 6:
        return "retrieval & ranking"
    if "data scien" in t:
        return "data-science / ML"
    return "ML engineering"


def _evidence(f):
    """A career-evidence clause, varied by the candidate's own signals."""
    spec = _specialization(f)
    has_prod = f["prod_hits"] >= 2
    has_eval = f["eval_hits"] >= 1
    deep = f["domain_hits"] >= 6
    if deep and has_prod and has_eval:
        return ("career history shows deep " + spec + " work shipped to "
                "production with offline/online evaluation rigor")
    if deep and has_prod:
        return "strong track record building " + spec + " systems in production"
    if has_prod and has_eval:
        return ("has shipped " + spec + " work to real users and measures it "
                "(offline/online metrics)")
    if has_prod:
        return "hands-on " + spec + " experience deployed in production"
    if f["domain_hits"] >= 2:
        return "demonstrated " + spec + " work across roles"
    return "relevant " + spec + " background"


def _engagement(f):
    """A short positive engagement note when the behavioral signals are good."""
    s = f.get("sig_summary") or {}
    rr = s.get("recruiter_response_rate")
    otw = s.get("open_to_work_flag")
    di = f.get("days_inactive", 999)
    bits = []
    if otw is True and di <= 60:
        bits.append("open to work and recently active")
    elif otw is True:
        bits.append("open to work")
    if rr is not None and rr >= 0.7:
        bits.append("high recruiter response rate (%.0f%%)" % (rr * 100))
    if not bits:
        return None
    return "; ".join(bits)


def _concerns(f, detail):
    out = []
    out += detail.get("dq_reasons", [])
    out += detail.get("exp_notes", [])
    out += detail.get("beh_notes", [])
    out += detail.get("loc_notes", [])
    if detail.get("honeypot_flags"):
        out.append("profile fails internal consistency checks")
    seen, dedup = set(), []
    for c in out:
        if c not in seen:
            seen.add(c)
            dedup.append(c)
    return dedup[:2]


def make_reasoning(f, detail, rank):
    lead = _lead(detail["final"])
    title = f["title"]
    yoe = f["yoe"]
    comp = f["company"]
    sent = "%s: %s (%.1fy) at %s - %s" % (lead, title, yoe, comp, _evidence(f))

    eng = _engagement(f)
    concerns = _concerns(f, detail)
    if concerns:
        sent += ". Concern: " + "; ".join(concerns) + "."
    elif eng and rank <= 60:
        sent += "; " + eng + "."
    else:
        sent += "."
    sent = re.sub(r"\s+", " ", sent).strip()
    return sent

from collections import Counter as _Counter

def run_ranker(candidates, top_n=20):
    df=_Counter(); n=0
    for c in candidates:
        n+=1
        df.update(set(tokenize(career_prose(c))))
    hi=0.5*n
    kept={t:d for t,d in df.items() if 1<=d<=hi}
    idf=build_idf(kept,n)
    qvec,qnorm=query_vector(idf)
    scored=[]
    for c in candidates:
        f=extract(c)
        sem=doc_cosine(tokenize(f["prose"]),idf,qvec,qnorm)
        final,detail=score_candidate(f,sem)
        scored.append((final,c["candidate_id"],f,detail))
    scored.sort(key=lambda x:(-x[0],x[1]))
    rows=[]; prev=None
    for i,(final,cid,f,detail) in enumerate(scored[:top_n]):
        s=round(final,6)
        if prev is not None and s>prev: s=prev
        prev=s
        rows.append({"candidate_id":cid,"rank":i+1,"score":"%.6f"%s,
                     "reasoning":make_reasoning(f,detail,i+1)})
    return rows



## 2. Embedded sample

A 47-candidate sample that includes honeypots (an NLP
Engineer and a Search Engineer with impossible tenure) and keyword-stuffer decoys
(non-engineering titles padded with AI skills). A keyword matcher ranks those near
the top; this system should not.


In [ ]:
SAMPLE_JSON = '[{"candidate_id": "CAND_0000001", "profile": {"anonymized_name": "Ira Vora", "headline": "Backend Engineer | SQL, Spark, Cloud", "summary": "Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I\'m a backend/data hybrid \\u2014 Spark, Airflow, SQL warehouses are home territory; I\'m building competence on the ML side. My toolkit is solid on the data engineering side \\u2014 Python, SQL, Spark, Airflow, warehouse design \\u2014 and I\'ve completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.", "location": "Toronto", "country": "Canada", "years_of_experience": 6.9, "current_title": "Backend Engineer", "current_company": "Mindtree", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Mindtree", "title": "Backend Engineer", "start_date": "2024-03-08", "end_date": null, "duration_months": 27, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Implemented streaming data pipelines on Kafka and Spark Streaming for a real-time user-activity processing platform. Designed the schema-registry integration, the watermark/state management approach, and the deduplication logic for late-arriving events. Worked closely with the data science team to make sure feature pipelines aligned with what their models needed. Most of my career has been data engineering, with some adjacent ML exposure."}, {"company": "Dunder Mifflin", "title": "Analytics Engineer", "start_date": "2019-07-03", "end_date": "2024-01-08", "duration_months": 55, "is_current": false, "industry": "Paper Products", "company_size": "201-500", "description": "Built and maintained data pipelines on Apache Airflow processing ~500GB of daily transactional data across 12 source systems. Worked extensively with Spark (PySpark) for batch processing and dbt for the transformation/modeling layer in our Snowflake warehouse. Owned the on-call rotation for data quality issues \\u2014 wrote most of the data quality checks that detect schema drift and unusual volume changes. The pipeline supports the analytics team and a few internal ML models."}], "education": [{"institution": "Lovely Professional University", "degree": "B.E.", "field_of_study": "Computer Science", "start_year": 2017, "end_year": 2020, "grade": "8.24 CGPA", "tier": "tier_3"}], "skills": [{"name": "Tailwind", "proficiency": "intermediate", "endorsements": 3, "duration_months": 13}, {"name": "NLP", "proficiency": "advanced", "endorsements": 37, "duration_months": 26}, {"name": "Image Classification", "proficiency": "advanced", "endorsements": 7, "duration_months": 40}, {"name": "Fine-tuning LLMs", "proficiency": "advanced", "endorsements": 21, "duration_months": 36}, {"name": "Weights & Biases", "proficiency": "intermediate", "endorsements": 13, "duration_months": 30}, {"name": "Speech Recognition", "proficiency": "advanced", "endorsements": 52, "duration_months": 33}, {"name": "Photoshop", "proficiency": "intermediate", "endorsements": 8, "duration_months": 24}, {"name": "TTS", "proficiency": "advanced", "endorsements": 56, "duration_months": 60}, {"name": "LoRA", "proficiency": "intermediate", "endorsements": 0, "duration_months": 28}, {"name": "Apache Beam", "proficiency": "intermediate", "endorsements": 4, "duration_months": 9}, {"name": "AWS", "proficiency": "beginner", "endorsements": 5, "duration_months": 8}, {"name": "Flask", "proficiency": "beginner", "endorsements": 15, "duration_months": 15}, {"name": "BentoML", "proficiency": "intermediate", "endorsements": 3, "duration_months": 36}, {"name": "Milvus", "proficiency": "advanced", "endorsements": 40, "duration_months": 35}, {"name": "GANs", "proficiency": "advanced", "endorsements": 12, "duration_months": 19}, {"name": "Statistical Modeling", "proficiency": "intermediate", "endorsements": 9, "duration_months": 8}, {"name": "GCP", "proficiency": "beginner", "endorsements": 7, "duration_months": 2}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 86.9, "signup_date": "2025-10-16", "last_active_date": "2026-05-20", "open_to_work_flag": true, "profile_views_received_30d": 23, "applications_submitted_30d": 2, "recruiter_response_rate": 0.34, "avg_response_time_hours": 177.8, "skill_assessment_scores": {"NLP": 38.8, "Image Classification": 64.8, "Fine-tuning LLMs": 41.6, "Speech Recognition": 53.7}, "connection_count": 356, "endorsements_received": 35, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 18.7, "max": 36.1}, "preferred_work_mode": "onsite", "willing_to_relocate": false, "github_activity_score": 9.2, "search_appearance_30d": 249, "saved_by_recruiters_30d": 4, "interview_completion_rate": 0.71, "offer_acceptance_rate": 0.58, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000002", "profile": {"anonymized_name": "Saanvi Sethi", "headline": "Operations Manager | 12.5+ yrs experience", "summary": "Professional with 12.5+ years of experience. My professional background is in marketing manager \\u2014 I\'ve built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Chennai, Tamil Nadu", "country": "India", "years_of_experience": 12.5, "current_title": "Operations Manager", "current_company": "Wipro", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Wipro", "title": "Operations Manager", "start_date": "2022-11-14", "end_date": null, "duration_months": 43, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}, {"company": "Wipro", "title": "Operations Manager", "start_date": "2021-09-13", "end_date": "2022-11-07", "duration_months": 14, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}, {"company": "Acme Corp", "title": "Marketing Manager", "start_date": "2017-03-08", "end_date": "2021-08-14", "duration_months": 54, "is_current": false, "industry": "Manufacturing", "company_size": "201-500", "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \\u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."}, {"company": "Dunder Mifflin", "title": "Marketing Manager", "start_date": "2014-01-23", "end_date": "2017-03-08", "duration_months": 38, "is_current": false, "industry": "Paper Products", "company_size": "201-500", "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."}], "education": [{"institution": "Local Engineering College", "degree": "B.Sc", "field_of_study": "Mathematics", "start_year": 2007, "end_year": 2011, "grade": "77%", "tier": "tier_4"}], "skills": [{"name": "Project Management", "proficiency": "intermediate", "endorsements": 14, "duration_months": 23}, {"name": "React", "proficiency": "intermediate", "endorsements": 6, "duration_months": 35}, {"name": "Photoshop", "proficiency": "intermediate", "endorsements": 9, "duration_months": 35}, {"name": "TypeScript", "proficiency": "beginner", "endorsements": 2, "duration_months": 3}, {"name": "Marketing", "proficiency": "beginner", "endorsements": 9, "duration_months": 11}, {"name": "Kafka", "proficiency": "intermediate", "endorsements": 3, "duration_months": 16}, {"name": "JavaScript", "proficiency": "beginner", "endorsements": 9, "duration_months": 3}, {"name": "Feature Engineering", "proficiency": "intermediate", "endorsements": 11, "duration_months": 27}, {"name": "GCP", "proficiency": "intermediate", "endorsements": 7, "duration_months": 30}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 78.7, "signup_date": "2025-07-28", "last_active_date": "2025-11-12", "open_to_work_flag": true, "profile_views_received_30d": 7, "applications_submitted_30d": 1, "recruiter_response_rate": 0.29, "avg_response_time_hours": 171.6, "skill_assessment_scores": {}, "connection_count": 179, "endorsements_received": 3, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 8.8, "max": 9.0}, "preferred_work_mode": "flexible", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 107, "saved_by_recruiters_30d": 10, "interview_completion_rate": 0.62, "offer_acceptance_rate": -1, "verified_email": false, "verified_phone": false, "linkedin_connected": false}}, {"candidate_id": "CAND_0000003", "profile": {"anonymized_name": "Yash Agarwal", "headline": "Customer Support | 1.1+ yrs experience", "summary": "Professional with 1.1+ years of experience. I\'ve spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Austin", "country": "USA", "years_of_experience": 1.1, "current_title": "Customer Support", "current_company": "TCS", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "TCS", "title": "Customer Support", "start_date": "2025-05-02", "end_date": null, "duration_months": 13, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}], "education": [{"institution": "Local Engineering College", "degree": "M.E.", "field_of_study": "Chemical Engineering", "start_year": 2005, "end_year": 2010, "grade": "6.82 CGPA", "tier": "tier_4"}, {"institution": "Chandigarh University", "degree": "M.Sc", "field_of_study": "Information Technology", "start_year": 2017, "end_year": 2021, "grade": "87%", "tier": "tier_3"}], "skills": [{"name": "Angular", "proficiency": "intermediate", "endorsements": 13, "duration_months": 10}, {"name": "SEO", "proficiency": "beginner", "endorsements": 11, "duration_months": 11}, {"name": "Excel", "proficiency": "intermediate", "endorsements": 2, "duration_months": 15}, {"name": "Accounting", "proficiency": "beginner", "endorsements": 7, "duration_months": 18}, {"name": "Kubernetes", "proficiency": "intermediate", "endorsements": 0, "duration_months": 34}, {"name": "Databricks", "proficiency": "beginner", "endorsements": 14, "duration_months": 18}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 31.9, "signup_date": "2024-08-02", "last_active_date": "2026-03-21", "open_to_work_flag": false, "profile_views_received_30d": 1, "applications_submitted_30d": 9, "recruiter_response_rate": 0.46, "avg_response_time_hours": 119.4, "skill_assessment_scores": {}, "connection_count": 19, "endorsements_received": 46, "notice_period_days": 150, "expected_salary_range_inr_lpa": {"min": 11.2, "max": 18.1}, "preferred_work_mode": "hybrid", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 28, "saved_by_recruiters_30d": 4, "interview_completion_rate": 0.86, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": false, "linkedin_connected": false}}, {"candidate_id": "CAND_0000004", "profile": {"anonymized_name": "Anil Bose", "headline": "Marketing Manager | Driving business outcomes", "summary": "Professional with 3.8+ years of experience. My professional background is in marketing manager \\u2014 I\'ve built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Sydney", "country": "Australia", "years_of_experience": 3.8, "current_title": "Marketing Manager", "current_company": "Dunder Mifflin", "current_company_size": "201-500", "current_industry": "Paper Products"}, "career_history": [{"company": "Dunder Mifflin", "title": "Marketing Manager", "start_date": "2025-04-02", "end_date": null, "duration_months": 14, "is_current": true, "industry": "Paper Products", "company_size": "201-500", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}, {"company": "Infosys", "title": "Operations Manager", "start_date": "2023-07-28", "end_date": "2025-03-19", "duration_months": 20, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \\u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."}, {"company": "Globex Inc", "title": "Business Analyst", "start_date": "2022-08-02", "end_date": "2023-05-29", "duration_months": 10, "is_current": false, "industry": "Manufacturing", "company_size": "501-1000", "description": "Operations management role at a logistics company. Owned daily fulfillment operations across 3 warehouses, managing a team of 80 across receiving, picking, packing, and outbound. Built and tracked the operational KPIs (on-time fulfillment, accuracy, cost per order) and led the continuous improvement initiatives that drove a 22% productivity gain over 18 months."}], "education": [{"institution": "Local Engineering College", "degree": "B.Tech", "field_of_study": "Machine Learning", "start_year": 2015, "end_year": 2019, "grade": "7.72 CGPA", "tier": "tier_4"}, {"institution": "Lovely Professional University", "degree": "Ph.D", "field_of_study": "Electronics", "start_year": 2013, "end_year": 2016, "grade": "7.61 CGPA", "tier": "tier_3"}], "skills": [{"name": "Node.js", "proficiency": "intermediate", "endorsements": 1, "duration_months": 20}, {"name": "Content Writing", "proficiency": "beginner", "endorsements": 7, "duration_months": 3}, {"name": "Redux", "proficiency": "beginner", "endorsements": 14, "duration_months": 8}, {"name": "Airflow", "proficiency": "intermediate", "endorsements": 11, "duration_months": 27}, {"name": "GraphQL", "proficiency": "beginner", "endorsements": 13, "duration_months": 2}, {"name": "Object Detection", "proficiency": "intermediate", "endorsements": 3, "duration_months": 17}, {"name": "Webpack", "proficiency": "beginner", "endorsements": 10, "duration_months": 7}, {"name": "Six Sigma", "proficiency": "beginner", "endorsements": 1, "duration_months": 12}, {"name": "SAP", "proficiency": "intermediate", "endorsements": 6, "duration_months": 9}, {"name": "JavaScript", "proficiency": "intermediate", "endorsements": 14, "duration_months": 29}], "certifications": [{"name": "AWS Certified Cloud Practitioner", "issuer": "AWS", "year": 2025}, {"name": "Scrum Master Certified", "issuer": "Scrum Alliance", "year": 2025}], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 28.5, "signup_date": "2025-07-21", "last_active_date": "2026-03-25", "open_to_work_flag": false, "profile_views_received_30d": 3, "applications_submitted_30d": 9, "recruiter_response_rate": 0.26, "avg_response_time_hours": 104.1, "skill_assessment_scores": {}, "connection_count": 485, "endorsements_received": 22, "notice_period_days": 120, "expected_salary_range_inr_lpa": {"min": 4.6, "max": 6.7}, "preferred_work_mode": "onsite", "willing_to_relocate": true, "github_activity_score": -1, "search_appearance_30d": 5, "saved_by_recruiters_30d": 8, "interview_completion_rate": 0.35, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": true}}, {"candidate_id": "CAND_0000005", "profile": {"anonymized_name": "Aisha Sethi", "headline": "Accountant | Helping teams scale", "summary": "Professional with 11.0+ years of experience. I\'ve spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Gurgaon, Haryana", "country": "India", "years_of_experience": 11.0, "current_title": "Accountant", "current_company": "Stark Industries", "current_company_size": "1001-5000", "current_industry": "Manufacturing"}, "career_history": [{"company": "Stark Industries", "title": "Accountant", "start_date": "2022-02-17", "end_date": null, "duration_months": 52, "is_current": true, "industry": "Manufacturing", "company_size": "1001-5000", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}, {"company": "Wipro", "title": "HR Manager", "start_date": "2018-05-25", "end_date": "2022-02-03", "duration_months": 45, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Senior accounting role at a mid-sized company \\u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."}, {"company": "Initech", "title": "HR Manager", "start_date": "2016-06-04", "end_date": "2018-05-25", "duration_months": 24, "is_current": false, "industry": "Software", "company_size": "51-200", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}, {"company": "TCS", "title": "Accountant", "start_date": "2015-09-08", "end_date": "2016-06-04", "duration_months": 9, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}], "education": [{"institution": "Chandigarh University", "degree": "M.Sc", "field_of_study": "Information Technology", "start_year": 2007, "end_year": 2012, "grade": "87%", "tier": "tier_3"}], "skills": [{"name": "SQL", "proficiency": "beginner", "endorsements": 12, "duration_months": 12}, {"name": "PowerPoint", "proficiency": "beginner", "endorsements": 6, "duration_months": 14}, {"name": "Photoshop", "proficiency": "beginner", "endorsements": 4, "duration_months": 18}, {"name": "Tailwind", "proficiency": "intermediate", "endorsements": 15, "duration_months": 35}, {"name": "Apache Flink", "proficiency": "intermediate", "endorsements": 1, "duration_months": 30}, {"name": "Image Classification", "proficiency": "advanced", "endorsements": 50, "duration_months": 38}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 84.6, "signup_date": "2023-10-07", "last_active_date": "2025-10-01", "open_to_work_flag": true, "profile_views_received_30d": 12, "applications_submitted_30d": 2, "recruiter_response_rate": 0.37, "avg_response_time_hours": 116.7, "skill_assessment_scores": {}, "connection_count": 300, "endorsements_received": 14, "notice_period_days": 30, "expected_salary_range_inr_lpa": {"min": 12.4, "max": 19.7}, "preferred_work_mode": "hybrid", "willing_to_relocate": true, "github_activity_score": -1, "search_appearance_30d": 67, "saved_by_recruiters_30d": 1, "interview_completion_rate": 0.74, "offer_acceptance_rate": -1, "verified_email": false, "verified_phone": true, "linkedin_connected": true}}, {"candidate_id": "CAND_0000006", "profile": {"anonymized_name": "Rajesh Desai", "headline": "Business Analyst | 6.0+ yrs experience", "summary": "Professional with 6.0+ years of experience. I\'ve spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Austin", "country": "USA", "years_of_experience": 6.0, "current_title": "Business Analyst", "current_company": "Wayne Enterprises", "current_company_size": "10001+", "current_industry": "Conglomerate"}, "career_history": [{"company": "Wayne Enterprises", "title": "Business Analyst", "start_date": "2023-09-10", "end_date": null, "duration_months": 33, "is_current": true, "industry": "Conglomerate", "company_size": "10001+", "description": "Senior accounting role at a mid-sized company \\u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."}, {"company": "Pied Piper", "title": "Mechanical Engineer", "start_date": "2020-07-27", "end_date": "2023-09-10", "duration_months": 38, "is_current": false, "industry": "Software", "company_size": "11-50", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}], "education": [{"institution": "Lovely Professional University", "degree": "B.Sc", "field_of_study": "Artificial Intelligence", "start_year": 2005, "end_year": 2008, "grade": "9.26 CGPA", "tier": "tier_3"}], "skills": [{"name": "Content Writing", "proficiency": "intermediate", "endorsements": 0, "duration_months": 33}, {"name": "SEO", "proficiency": "intermediate", "endorsements": 13, "duration_months": 31}, {"name": "Redux", "proficiency": "beginner", "endorsements": 15, "duration_months": 12}, {"name": "SQL", "proficiency": "beginner", "endorsements": 9, "duration_months": 11}, {"name": "Sales", "proficiency": "intermediate", "endorsements": 5, "duration_months": 27}, {"name": "gRPC", "proficiency": "beginner", "endorsements": 8, "duration_months": 3}, {"name": "Django", "proficiency": "intermediate", "endorsements": 3, "duration_months": 11}, {"name": "Terraform", "proficiency": "beginner", "endorsements": 4, "duration_months": 13}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 29.7, "signup_date": "2026-04-26", "last_active_date": "2026-02-28", "open_to_work_flag": false, "profile_views_received_30d": 53, "applications_submitted_30d": 8, "recruiter_response_rate": 0.12, "avg_response_time_hours": 172.1, "skill_assessment_scores": {}, "connection_count": 389, "endorsements_received": 29, "notice_period_days": 150, "expected_salary_range_inr_lpa": {"min": 7.7, "max": 11.7}, "preferred_work_mode": "remote", "willing_to_relocate": true, "github_activity_score": -1, "search_appearance_30d": 131, "saved_by_recruiters_30d": 9, "interview_completion_rate": 0.57, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000007", "profile": {"anonymized_name": "Vihaan Bose", "headline": "Civil Engineer | 5.5+ yrs experience", "summary": "Professional with 5.5+ years of experience. My professional background is in marketing manager \\u2014 I\'ve built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Gurgaon, Haryana", "country": "India", "years_of_experience": 5.5, "current_title": "Civil Engineer", "current_company": "Wipro", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Wipro", "title": "Civil Engineer", "start_date": "2023-04-13", "end_date": null, "duration_months": 38, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."}, {"company": "Initech", "title": "Mechanical Engineer", "start_date": "2021-01-16", "end_date": "2023-04-06", "duration_months": 27, "is_current": false, "industry": "Software", "company_size": "51-200", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}], "education": [{"institution": "SRM University", "degree": "M.E.", "field_of_study": "Data Science", "start_year": 2009, "end_year": 2013, "grade": "8.28 CGPA", "tier": "tier_2"}], "skills": [{"name": "Content Writing", "proficiency": "beginner", "endorsements": 12, "duration_months": 14}, {"name": "MongoDB", "proficiency": "intermediate", "endorsements": 13, "duration_months": 9}, {"name": "Sales", "proficiency": "intermediate", "endorsements": 0, "duration_months": 27}, {"name": "Spark", "proficiency": "beginner", "endorsements": 14, "duration_months": 14}, {"name": "Scrum", "proficiency": "beginner", "endorsements": 4, "duration_months": 18}, {"name": "Apache Beam", "proficiency": "beginner", "endorsements": 4, "duration_months": 3}, {"name": "Illustrator", "proficiency": "beginner", "endorsements": 14, "duration_months": 2}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 74.6, "signup_date": "2025-09-29", "last_active_date": "2026-05-25", "open_to_work_flag": false, "profile_views_received_30d": 2, "applications_submitted_30d": 1, "recruiter_response_rate": 0.62, "avg_response_time_hours": 61.3, "skill_assessment_scores": {}, "connection_count": 122, "endorsements_received": 50, "notice_period_days": 30, "expected_salary_range_inr_lpa": {"min": 6.7, "max": 14.6}, "preferred_work_mode": "onsite", "willing_to_relocate": true, "github_activity_score": -1, "search_appearance_30d": 104, "saved_by_recruiters_30d": 8, "interview_completion_rate": 0.47, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": true}}, {"candidate_id": "CAND_0000008", "profile": {"anonymized_name": "Shaurya Chatterjee", "headline": "Operations Manager | 3.6+ yrs experience", "summary": "Professional with 3.6+ years of experience. I\'ve spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Noida, Uttar Pradesh", "country": "India", "years_of_experience": 3.6, "current_title": "Operations Manager", "current_company": "Wipro", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Wipro", "title": "Operations Manager", "start_date": "2022-11-14", "end_date": null, "duration_months": 43, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \\u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}], "education": [{"institution": "Anna University", "degree": "B.Tech", "field_of_study": "Data Science", "start_year": 2008, "end_year": 2012, "grade": "8.60 CGPA", "tier": "tier_2"}, {"institution": "SRM University", "degree": "M.Sc", "field_of_study": "Computer Engineering", "start_year": 2009, "end_year": 2013, "grade": "67%", "tier": "tier_2"}], "skills": [{"name": "Java", "proficiency": "intermediate", "endorsements": 2, "duration_months": 32}, {"name": "BigQuery", "proficiency": "beginner", "endorsements": 5, "duration_months": 9}, {"name": "Spark", "proficiency": "beginner", "endorsements": 4, "duration_months": 6}, {"name": "Accounting", "proficiency": "beginner", "endorsements": 8, "duration_months": 3}, {"name": "Kubernetes", "proficiency": "intermediate", "endorsements": 2, "duration_months": 9}, {"name": "TypeScript", "proficiency": "intermediate", "endorsements": 14, "duration_months": 11}, {"name": "Rust", "proficiency": "intermediate", "endorsements": 12, "duration_months": 16}, {"name": "HTML", "proficiency": "beginner", "endorsements": 8, "duration_months": 11}], "certifications": [{"name": "Six Sigma Green Belt", "issuer": "ASQ", "year": 2018}], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 63.0, "signup_date": "2022-06-26", "last_active_date": "2025-12-13", "open_to_work_flag": false, "profile_views_received_30d": 28, "applications_submitted_30d": 5, "recruiter_response_rate": 0.42, "avg_response_time_hours": 98.4, "skill_assessment_scores": {}, "connection_count": 285, "endorsements_received": 7, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 6.6, "max": 17.2}, "preferred_work_mode": "onsite", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 91, "saved_by_recruiters_30d": 0, "interview_completion_rate": 0.74, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": false, "linkedin_connected": false}}, {"candidate_id": "CAND_0000009", "profile": {"anonymized_name": "Amit Shah", "headline": "Mechanical Engineer | Driving business outcomes", "summary": "Professional with 11.0+ years of experience. My professional background is in marketing manager \\u2014 I\'ve built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "New York", "country": "USA", "years_of_experience": 11.0, "current_title": "Mechanical Engineer", "current_company": "Dunder Mifflin", "current_company_size": "201-500", "current_industry": "Paper Products"}, "career_history": [{"company": "Dunder Mifflin", "title": "Mechanical Engineer", "start_date": "2022-10-15", "end_date": null, "duration_months": 44, "is_current": true, "industry": "Paper Products", "company_size": "201-500", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}, {"company": "Wipro", "title": "Content Writer", "start_date": "2021-02-22", "end_date": "2022-10-15", "duration_months": 20, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \\u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}, {"company": "Stark Industries", "title": "Customer Support", "start_date": "2017-02-13", "end_date": "2021-02-22", "duration_months": 49, "is_current": false, "industry": "Manufacturing", "company_size": "1001-5000", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}, {"company": "Acme Corp", "title": "Project Manager", "start_date": "2015-08-23", "end_date": "2017-02-13", "duration_months": 18, "is_current": false, "industry": "Manufacturing", "company_size": "201-500", "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."}], "education": [{"institution": "KIIT University", "degree": "B.Tech", "field_of_study": "Electronics", "start_year": 2009, "end_year": 2014, "grade": "7.89 CGPA", "tier": "tier_3"}], "skills": [{"name": "Snowflake", "proficiency": "intermediate", "endorsements": 5, "duration_months": 8}, {"name": "gRPC", "proficiency": "beginner", "endorsements": 6, "duration_months": 12}, {"name": "JavaScript", "proficiency": "intermediate", "endorsements": 0, "duration_months": 23}, {"name": "OpenCV", "proficiency": "intermediate", "endorsements": 12, "duration_months": 36}, {"name": "Go", "proficiency": "intermediate", "endorsements": 12, "duration_months": 20}, {"name": "PowerPoint", "proficiency": "intermediate", "endorsements": 1, "duration_months": 10}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 39.6, "signup_date": "2025-10-19", "last_active_date": "2026-01-27", "open_to_work_flag": false, "profile_views_received_30d": 50, "applications_submitted_30d": 8, "recruiter_response_rate": 0.53, "avg_response_time_hours": 202.0, "skill_assessment_scores": {}, "connection_count": 516, "endorsements_received": 34, "notice_period_days": 150, "expected_salary_range_inr_lpa": {"min": 16.0, "max": 7.3}, "preferred_work_mode": "remote", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 74, "saved_by_recruiters_30d": 1, "interview_completion_rate": 0.54, "offer_acceptance_rate": 0.48, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000010", "profile": {"anonymized_name": "Aarav Kapoor", "headline": "Data Engineer | Data pipelines & analytics", "summary": "Software / data professional with 4.6 years of experience building data pipelines, backend systems, and analytics infrastructure. Most of my work has been data pipelines and analytics infrastructure; I\'ve shipped a couple of small ML features but it\'s not the core of my day. My toolkit is solid on the data engineering side \\u2014 Python, SQL, Spark, Airflow, warehouse design \\u2014 and I\'ve completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.", "location": "London", "country": "UK", "years_of_experience": 4.6, "current_title": "Data Engineer", "current_company": "Ola", "current_company_size": "5001-10000", "current_industry": "Transportation"}, "career_history": [{"company": "Ola", "title": "Data Engineer", "start_date": "2021-11-19", "end_date": null, "duration_months": 55, "is_current": true, "industry": "Transportation", "company_size": "5001-10000", "description": "Mixed data science and analytics-engineering role at a marketing-analytics startup. Spent maybe 30% of my time on lightweight ML (clustering, classification, churn prediction in sklearn/XGBoost) and 70% on data infrastructure and dashboards. Comfortable with the modeling work but I wouldn\'t call myself an ML specialist. Built our experimentation framework that supports the product team\'s A/B tests."}], "education": [{"institution": "Generic State University", "degree": "B.E.", "field_of_study": "Mathematics", "start_year": 2007, "end_year": 2011, "grade": "85%", "tier": "tier_4"}, {"institution": "Local Engineering College", "degree": "M.S.", "field_of_study": "Computer Engineering", "start_year": 2013, "end_year": 2018, "grade": "7.73 CGPA", "tier": "tier_4"}], "skills": [{"name": "GCP", "proficiency": "beginner", "endorsements": 7, "duration_months": 8}, {"name": "Spring Boot", "proficiency": "beginner", "endorsements": 3, "duration_months": 2}, {"name": "Kubeflow", "proficiency": "intermediate", "endorsements": 11, "duration_months": 19}, {"name": "Java", "proficiency": "intermediate", "endorsements": 12, "duration_months": 19}, {"name": "GANs", "proficiency": "advanced", "endorsements": 58, "duration_months": 57}, {"name": "Figma", "proficiency": "beginner", "endorsements": 4, "duration_months": 3}, {"name": "Elasticsearch", "proficiency": "intermediate", "endorsements": 15, "duration_months": 17}, {"name": "OpenCV", "proficiency": "advanced", "endorsements": 0, "duration_months": 24}, {"name": "CNN", "proficiency": "intermediate", "endorsements": 15, "duration_months": 8}, {"name": "Azure", "proficiency": "beginner", "endorsements": 7, "duration_months": 11}, {"name": "Prompt Engineering", "proficiency": "advanced", "endorsements": 42, "duration_months": 35}, {"name": "Object Detection", "proficiency": "advanced", "endorsements": 55, "duration_months": 58}, {"name": "MLOps", "proficiency": "intermediate", "endorsements": 3, "duration_months": 10}, {"name": "Python", "proficiency": "intermediate", "endorsements": 7, "duration_months": 14}, {"name": "BM25", "proficiency": "advanced", "endorsements": 55, "duration_months": 55}, {"name": "Weights & Biases", "proficiency": "advanced", "endorsements": 4, "duration_months": 21}, {"name": "Sales", "proficiency": "beginner", "endorsements": 5, "duration_months": 18}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 81.6, "signup_date": "2026-01-09", "last_active_date": "2026-04-29", "open_to_work_flag": false, "profile_views_received_30d": 60, "applications_submitted_30d": 13, "recruiter_response_rate": 0.4, "avg_response_time_hours": 19.0, "skill_assessment_scores": {"GANs": 53.3, "OpenCV": 65.5, "Prompt Engineering": 73.8, "Object Detection": 81.3}, "connection_count": 712, "endorsements_received": 38, "notice_period_days": 120, "expected_salary_range_inr_lpa": {"min": 13.0, "max": 32.0}, "preferred_work_mode": "hybrid", "willing_to_relocate": false, "github_activity_score": 33.7, "search_appearance_30d": 256, "saved_by_recruiters_30d": 2, "interview_completion_rate": 0.53, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000011", "profile": {"anonymized_name": "Deepak Desai", "headline": "QA Engineer | Cloud & DevOps", "summary": "Software engineer with 2.0 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. Most of my work is conventional backend engineering \\u2014 APIs, databases, queues, deployments. I\'ve been keeping up with AI/ML at a self-learner level \\u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \\u2014 but I haven\'t done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "Hyderabad, Telangana", "country": "India", "years_of_experience": 2.0, "current_title": "QA Engineer", "current_company": "Pied Piper", "current_company_size": "11-50", "current_industry": "Software"}, "career_history": [{"company": "Pied Piper", "title": "QA Engineer", "start_date": "2025-05-02", "end_date": null, "duration_months": 13, "is_current": true, "industry": "Software", "company_size": "11-50", "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."}, {"company": "Hooli", "title": "QA Engineer", "start_date": "2024-06-06", "end_date": "2025-04-02", "duration_months": 10, "is_current": false, "industry": "Software", "company_size": "1001-5000", "description": "Test automation and QA engineering for a fintech product. Built and maintained the end-to-end test suite using Selenium and pytest, plus the load-testing setup using Locust. Worked closely with developers on testability patterns and with product on acceptance criteria. Recent work has been on shifting test responsibility into the dev team \\u2014 moving from QA-as-gate to QA-as-coach. Career has been entirely in QA/test engineering."}], "education": [{"institution": "Chandigarh University", "degree": "B.Tech", "field_of_study": "Data Science", "start_year": 2014, "end_year": 2019, "grade": "6.96 CGPA", "tier": "tier_3"}, {"institution": "Anna University", "degree": "B.Sc", "field_of_study": "Information Technology", "start_year": 2015, "end_year": 2020, "grade": "9.16 CGPA", "tier": "tier_2"}], "skills": [{"name": "Recommendation Systems", "proficiency": "advanced", "endorsements": 5, "duration_months": 40}, {"name": "Scrum", "proficiency": "beginner", "endorsements": 13, "duration_months": 7}, {"name": "FastAPI", "proficiency": "beginner", "endorsements": 4, "duration_months": 12}, {"name": "Hugging Face Transformers", "proficiency": "intermediate", "endorsements": 1, "duration_months": 30}, {"name": "AWS", "proficiency": "beginner", "endorsements": 4, "duration_months": 18}, {"name": "Snowflake", "proficiency": "beginner", "endorsements": 4, "duration_months": 11}, {"name": "Spring Boot", "proficiency": "intermediate", "endorsements": 12, "duration_months": 31}, {"name": "PostgreSQL", "proficiency": "intermediate", "endorsements": 7, "duration_months": 24}, {"name": "Kubeflow", "proficiency": "advanced", "endorsements": 6, "duration_months": 59}, {"name": "Azure", "proficiency": "beginner", "endorsements": 12, "duration_months": 8}], "certifications": [{"name": "AWS Certified Cloud Practitioner", "issuer": "AWS", "year": 2019}, {"name": "Six Sigma Green Belt", "issuer": "ASQ", "year": 2021}], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 59.2, "signup_date": "2023-07-22", "last_active_date": "2026-01-19", "open_to_work_flag": false, "profile_views_received_30d": 112, "applications_submitted_30d": 0, "recruiter_response_rate": 0.56, "avg_response_time_hours": 184.4, "skill_assessment_scores": {"Recommendation Systems": 29.8}, "connection_count": 496, "endorsements_received": 9, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 15.5, "max": 13.9}, "preferred_work_mode": "flexible", "willing_to_relocate": false, "github_activity_score": 32.3, "search_appearance_30d": 200, "saved_by_recruiters_30d": 13, "interview_completion_rate": 0.45, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": true}}, {"candidate_id": "CAND_0000012", "profile": {"anonymized_name": "Anjali Krishnan", "headline": "Operations Manager | Driving business outcomes", "summary": "Professional with 1.1+ years of experience. My professional background is in marketing manager \\u2014 I\'ve built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Chandigarh, Chandigarh", "country": "India", "years_of_experience": 1.1, "current_title": "Operations Manager", "current_company": "Stark Industries", "current_company_size": "1001-5000", "current_industry": "Manufacturing"}, "career_history": [{"company": "Stark Industries", "title": "Operations Manager", "start_date": "2025-05-02", "end_date": null, "duration_months": 13, "is_current": true, "industry": "Manufacturing", "company_size": "1001-5000", "description": "Senior accounting role at a mid-sized company \\u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."}], "education": [{"institution": "Symbiosis International", "degree": "B.Sc", "field_of_study": "Physics", "start_year": 2018, "end_year": 2022, "grade": "68%", "tier": "tier_3"}, {"institution": "Christ University", "degree": "B.Sc", "field_of_study": "Mechanical Engineering", "start_year": 2011, "end_year": 2015, "grade": "7.28 CGPA", "tier": "tier_3"}], "skills": [{"name": "Azure", "proficiency": "beginner", "endorsements": 7, "duration_months": 10}, {"name": "Airflow", "proficiency": "intermediate", "endorsements": 2, "duration_months": 15}, {"name": "AWS", "proficiency": "intermediate", "endorsements": 15, "duration_months": 30}, {"name": "gRPC", "proficiency": "beginner", "endorsements": 9, "duration_months": 2}, {"name": "Vue.js", "proficiency": "intermediate", "endorsements": 2, "duration_months": 15}, {"name": "dbt", "proficiency": "intermediate", "endorsements": 11, "duration_months": 22}, {"name": "Agile", "proficiency": "intermediate", "endorsements": 4, "duration_months": 14}, {"name": "PowerPoint", "proficiency": "intermediate", "endorsements": 1, "duration_months": 27}, {"name": "Content Writing", "proficiency": "intermediate", "endorsements": 3, "duration_months": 36}, {"name": "Project Management", "proficiency": "beginner", "endorsements": 5, "duration_months": 3}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 53.4, "signup_date": "2024-01-28", "last_active_date": "2025-10-28", "open_to_work_flag": false, "profile_views_received_30d": 60, "applications_submitted_30d": 1, "recruiter_response_rate": 0.16, "avg_response_time_hours": 38.4, "skill_assessment_scores": {}, "connection_count": 165, "endorsements_received": 31, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 14.8, "max": 7.9}, "preferred_work_mode": "flexible", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 61, "saved_by_recruiters_30d": 3, "interview_completion_rate": 0.42, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": false, "linkedin_connected": false}}, {"candidate_id": "CAND_0000013", "profile": {"anonymized_name": "Pari Nair", "headline": "Civil Engineer | Driving business outcomes", "summary": "Professional with 1.1+ years of experience. I\'m a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Dubai", "country": "UAE", "years_of_experience": 1.1, "current_title": "Civil Engineer", "current_company": "Globex Inc", "current_company_size": "501-1000", "current_industry": "Manufacturing"}, "career_history": [{"company": "Globex Inc", "title": "Civil Engineer", "start_date": "2025-05-02", "end_date": null, "duration_months": 13, "is_current": true, "industry": "Manufacturing", "company_size": "501-1000", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}], "education": [{"institution": "Delhi College of Engineering", "degree": "B.E.", "field_of_study": "Information Technology", "start_year": 2019, "end_year": 2022, "grade": "8.84 CGPA", "tier": "tier_2"}, {"institution": "Amity University", "degree": "Ph.D", "field_of_study": "Information Technology", "start_year": 2008, "end_year": 2013, "grade": "8.29 CGPA", "tier": "tier_3"}], "skills": [{"name": "React", "proficiency": "intermediate", "endorsements": 5, "duration_months": 23}, {"name": "Redux", "proficiency": "intermediate", "endorsements": 11, "duration_months": 9}, {"name": "Vue.js", "proficiency": "beginner", "endorsements": 12, "duration_months": 12}, {"name": "Six Sigma", "proficiency": "beginner", "endorsements": 1, "duration_months": 2}, {"name": "Spring Boot", "proficiency": "beginner", "endorsements": 12, "duration_months": 12}, {"name": "Spark", "proficiency": "intermediate", "endorsements": 7, "duration_months": 30}, {"name": "Data Pipelines", "proficiency": "intermediate", "endorsements": 6, "duration_months": 36}, {"name": "GCP", "proficiency": "intermediate", "endorsements": 0, "duration_months": 18}, {"name": "Flask", "proficiency": "intermediate", "endorsements": 12, "duration_months": 8}, {"name": "Snowflake", "proficiency": "intermediate", "endorsements": 2, "duration_months": 8}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 44.2, "signup_date": "2024-06-14", "last_active_date": "2026-03-26", "open_to_work_flag": true, "profile_views_received_30d": 16, "applications_submitted_30d": 3, "recruiter_response_rate": 0.28, "avg_response_time_hours": 80.3, "skill_assessment_scores": {}, "connection_count": 281, "endorsements_received": 9, "notice_period_days": 30, "expected_salary_range_inr_lpa": {"min": 11.6, "max": 8.1}, "preferred_work_mode": "remote", "willing_to_relocate": false, "github_activity_score": 35.6, "search_appearance_30d": 40, "saved_by_recruiters_30d": 12, "interview_completion_rate": 0.33, "offer_acceptance_rate": 0.26, "verified_email": true, "verified_phone": false, "linkedin_connected": false}}, {"candidate_id": "CAND_0000014", "profile": {"anonymized_name": "Atharv Joshi", "headline": "Frontend Engineer | Full-stack development", "summary": "Software engineer with 8.4 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. My background is full-stack, but my comfort zone is the backend and the database. I\'ve been keeping up with AI/ML at a self-learner level \\u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \\u2014 but I haven\'t done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "Hyderabad, Telangana", "country": "India", "years_of_experience": 8.4, "current_title": "Frontend Engineer", "current_company": "Zomato", "current_company_size": "5001-10000", "current_industry": "Food Delivery"}, "career_history": [{"company": "Zomato", "title": "Frontend Engineer", "start_date": "2023-09-10", "end_date": null, "duration_months": 33, "is_current": true, "industry": "Food Delivery", "company_size": "5001-10000", "description": "Frontend engineering at a media company. React, TypeScript, and the typical surrounding tooling (Webpack, Jest, Cypress). Built the company\'s design system from scratch and led the migration from a legacy AngularJS app. Strong on the frontend craft \\u2014 accessibility, performance, animations \\u2014 but limited backend exposure."}, {"company": "Dunder Mifflin", "title": "Software Engineer", "start_date": "2019-10-01", "end_date": "2023-09-10", "duration_months": 48, "is_current": false, "industry": "Paper Products", "company_size": "201-500", "description": "Test automation and QA engineering for a fintech product. Built and maintained the end-to-end test suite using Selenium and pytest, plus the load-testing setup using Locust. Worked closely with developers on testability patterns and with product on acceptance criteria. Recent work has been on shifting test responsibility into the dev team \\u2014 moving from QA-as-gate to QA-as-coach. Career has been entirely in QA/test engineering."}, {"company": "Acme Corp", "title": "Java Developer", "start_date": "2018-03-03", "end_date": "2019-09-24", "duration_months": 19, "is_current": false, "industry": "Manufacturing", "company_size": "201-500", "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."}], "education": [{"institution": "Lovely Professional University", "degree": "B.E.", "field_of_study": "Statistics", "start_year": 2012, "end_year": 2015, "grade": "7.45 CGPA", "tier": "tier_3"}], "skills": [{"name": "FAISS", "proficiency": "advanced", "endorsements": 40, "duration_months": 44}, {"name": "BigQuery", "proficiency": "intermediate", "endorsements": 6, "duration_months": 24}, {"name": "React", "proficiency": "beginner", "endorsements": 11, "duration_months": 10}, {"name": "OpenSearch", "proficiency": "advanced", "endorsements": 47, "duration_months": 59}, {"name": "OpenCV", "proficiency": "advanced", "endorsements": 30, "duration_months": 60}, {"name": "YOLO", "proficiency": "advanced", "endorsements": 1, "duration_months": 44}, {"name": "SAP", "proficiency": "intermediate", "endorsements": 5, "duration_months": 30}, {"name": "SEO", "proficiency": "beginner", "endorsements": 0, "duration_months": 12}, {"name": "REST APIs", "proficiency": "beginner", "endorsements": 3, "duration_months": 4}, {"name": "GANs", "proficiency": "advanced", "endorsements": 9, "duration_months": 33}, {"name": "dbt", "proficiency": "beginner", "endorsements": 0, "duration_months": 13}, {"name": "Photoshop", "proficiency": "intermediate", "endorsements": 1, "duration_months": 32}, {"name": "Tailwind", "proficiency": "intermediate", "endorsements": 2, "duration_months": 32}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "native"}], "redrob_signals": {"profile_completeness_score": 61.9, "signup_date": "2025-04-29", "last_active_date": "2026-04-12", "open_to_work_flag": false, "profile_views_received_30d": 21, "applications_submitted_30d": 1, "recruiter_response_rate": 0.8, "avg_response_time_hours": 7.7, "skill_assessment_scores": {"FAISS": 77.6}, "connection_count": 708, "endorsements_received": 63, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 9.0, "max": 30.0}, "preferred_work_mode": "remote", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 12, "saved_by_recruiters_30d": 0, "interview_completion_rate": 0.55, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000015", "profile": {"anonymized_name": "Rahul Agarwal", "headline": "Software Engineer | Cloud & DevOps", "summary": "Software engineer with 5.4 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. I\'ve worked across web frontends, REST APIs, and cloud deployments; comfortable in most parts of a typical SaaS stack. I\'ve been keeping up with AI/ML at a self-learner level \\u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \\u2014 but I haven\'t done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "Trivandrum, Kerala", "country": "India", "years_of_experience": 5.4, "current_title": "Software Engineer", "current_company": "Razorpay", "current_company_size": "1001-5000", "current_industry": "Fintech"}, "career_history": [{"company": "Razorpay", "title": "Software Engineer", "start_date": "2023-11-09", "end_date": null, "duration_months": 31, "is_current": true, "industry": "Fintech", "company_size": "1001-5000", "description": "Cloud infrastructure and DevOps work at an enterprise SaaS company. Owned the AWS account architecture (VPC, IAM, networking), the Terraform modules for our service deployments, and the Kubernetes cluster operations. Designed the CI/CD pipelines (GitLab CI + ArgoCD) and the monitoring stack (Prometheus, Grafana, Loki). Strong on the infra and ops side; haven\'t done much application development."}, {"company": "Hooli", "title": "Mobile Developer", "start_date": "2021-11-12", "end_date": "2023-11-02", "duration_months": 24, "is_current": false, "industry": "Software", "company_size": "1001-5000", "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."}, {"company": "Globex Inc", "title": "DevOps Engineer", "start_date": "2021-02-15", "end_date": "2021-11-12", "duration_months": 9, "is_current": false, "industry": "Manufacturing", "company_size": "501-1000", "description": "Java backend development at a large enterprise \\u2014 Spring Boot microservices, Kafka for inter-service messaging, Postgres + Redis for storage. Worked on the customer onboarding flow which involved orchestrating multiple downstream services. Solid on the Spring ecosystem, transaction handling, and the operational side of Java services. Looking to either go deeper on distributed systems or expand into modern application stacks."}], "education": [{"institution": "Local Engineering College", "degree": "Ph.D", "field_of_study": "Mathematics", "start_year": 2013, "end_year": 2017, "grade": "8.15 CGPA", "tier": "tier_4"}], "skills": [{"name": "PyTorch", "proficiency": "intermediate", "endorsements": 10, "duration_months": 15}, {"name": "Content Writing", "proficiency": "intermediate", "endorsements": 8, "duration_months": 31}, {"name": "Weights & Biases", "proficiency": "intermediate", "endorsements": 5, "duration_months": 24}, {"name": "Qdrant", "proficiency": "intermediate", "endorsements": 13, "duration_months": 9}, {"name": "Sales", "proficiency": "intermediate", "endorsements": 13, "duration_months": 30}, {"name": "Figma", "proficiency": "intermediate", "endorsements": 9, "duration_months": 29}, {"name": "BigQuery", "proficiency": "beginner", "endorsements": 0, "duration_months": 7}, {"name": "Computer Vision", "proficiency": "intermediate", "endorsements": 6, "duration_months": 20}, {"name": "Next.js", "proficiency": "intermediate", "endorsements": 12, "duration_months": 35}, {"name": "SEO", "proficiency": "intermediate", "endorsements": 10, "duration_months": 29}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 79.4, "signup_date": "2023-02-16", "last_active_date": "2026-02-12", "open_to_work_flag": true, "profile_views_received_30d": 93, "applications_submitted_30d": 3, "recruiter_response_rate": 0.32, "avg_response_time_hours": 117.7, "skill_assessment_scores": {}, "connection_count": 361, "endorsements_received": 61, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 21.8, "max": 18.9}, "preferred_work_mode": "remote", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 164, "saved_by_recruiters_30d": 8, "interview_completion_rate": 0.72, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000016", "profile": {"anonymized_name": "Aanya Malhotra", "headline": "Accountant | Helping teams scale", "summary": "Professional with 5.3+ years of experience. I\'m a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Gurgaon, Haryana", "country": "India", "years_of_experience": 5.3, "current_title": "Accountant", "current_company": "Infosys", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Infosys", "title": "Accountant", "start_date": "2024-12-03", "end_date": null, "duration_months": 18, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."}, {"company": "TCS", "title": "Mechanical Engineer", "start_date": "2021-09-06", "end_date": "2024-11-19", "duration_months": 39, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}, {"company": "Globex Inc", "title": "Operations Manager", "start_date": "2021-02-08", "end_date": "2021-08-07", "duration_months": 6, "is_current": false, "industry": "Manufacturing", "company_size": "501-1000", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}], "education": [{"institution": "Christ University", "degree": "B.E.", "field_of_study": "Electronics", "start_year": 2001, "end_year": 2006, "grade": "8.32 CGPA", "tier": "tier_3"}], "skills": [{"name": "Node.js", "proficiency": "beginner", "endorsements": 4, "duration_months": 11}, {"name": "Figma", "proficiency": "beginner", "endorsements": 12, "duration_months": 6}, {"name": "Data Pipelines", "proficiency": "beginner", "endorsements": 2, "duration_months": 7}, {"name": "Go", "proficiency": "beginner", "endorsements": 0, "duration_months": 10}, {"name": "Photoshop", "proficiency": "intermediate", "endorsements": 10, "duration_months": 22}, {"name": "Kubeflow", "proficiency": "advanced", "endorsements": 2, "duration_months": 54}, {"name": "Accounting", "proficiency": "intermediate", "endorsements": 4, "duration_months": 31}, {"name": "SQL", "proficiency": "intermediate", "endorsements": 14, "duration_months": 16}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 69.4, "signup_date": "2022-12-12", "last_active_date": "2025-12-21", "open_to_work_flag": true, "profile_views_received_30d": 76, "applications_submitted_30d": 5, "recruiter_response_rate": 0.64, "avg_response_time_hours": 205.6, "skill_assessment_scores": {}, "connection_count": 148, "endorsements_received": 2, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 6.1, "max": 8.1}, "preferred_work_mode": "hybrid", "willing_to_relocate": false, "github_activity_score": 42.9, "search_appearance_30d": 126, "saved_by_recruiters_30d": 5, "interview_completion_rate": 0.66, "offer_acceptance_rate": -1, "verified_email": false, "verified_phone": false, "linkedin_connected": true}}, {"candidate_id": "CAND_0000017", "profile": {"anonymized_name": "Anil Pandey", "headline": "Accountant | 12.3+ yrs experience", "summary": "Professional with 12.3+ years of experience. My professional background is in marketing manager \\u2014 I\'ve built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Bangalore, Karnataka", "country": "India", "years_of_experience": 12.3, "current_title": "Accountant", "current_company": "Wipro", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Wipro", "title": "Accountant", "start_date": "2024-03-08", "end_date": null, "duration_months": 27, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}, {"company": "Infosys", "title": "Customer Support", "start_date": "2021-02-08", "end_date": "2024-02-23", "duration_months": 37, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}, {"company": "Initech", "title": "Mechanical Engineer", "start_date": "2017-07-29", "end_date": "2021-02-08", "duration_months": 43, "is_current": false, "industry": "Software", "company_size": "51-200", "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."}, {"company": "Acme Corp", "title": "Accountant", "start_date": "2014-05-16", "end_date": "2017-07-29", "duration_months": 39, "is_current": false, "industry": "Manufacturing", "company_size": "201-500", "description": "Senior accounting role at a mid-sized company \\u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."}], "education": [{"institution": "Tier-3 Engineering College", "degree": "M.Tech", "field_of_study": "Data Science", "start_year": 2017, "end_year": 2022, "grade": "7.58 CGPA", "tier": "tier_4"}], "skills": [{"name": "Next.js", "proficiency": "beginner", "endorsements": 9, "duration_months": 18}, {"name": "Java", "proficiency": "intermediate", "endorsements": 13, "duration_months": 15}, {"name": "Apache Flink", "proficiency": "intermediate", "endorsements": 4, "duration_months": 16}, {"name": "Sales", "proficiency": "intermediate", "endorsements": 4, "duration_months": 8}, {"name": "Tally", "proficiency": "intermediate", "endorsements": 14, "duration_months": 12}, {"name": "PostgreSQL", "proficiency": "intermediate", "endorsements": 11, "duration_months": 25}, {"name": "REST APIs", "proficiency": "beginner", "endorsements": 15, "duration_months": 4}, {"name": "Hadoop", "proficiency": "intermediate", "endorsements": 4, "duration_months": 35}], "certifications": [{"name": "Six Sigma Green Belt", "issuer": "ASQ", "year": 2018}, {"name": "Scrum Master Certified", "issuer": "Scrum Alliance", "year": 2022}], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 38.7, "signup_date": "2025-08-11", "last_active_date": "2025-11-04", "open_to_work_flag": false, "profile_views_received_30d": 3, "applications_submitted_30d": 4, "recruiter_response_rate": 0.27, "avg_response_time_hours": 197.4, "skill_assessment_scores": {}, "connection_count": 35, "endorsements_received": 23, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 13.8, "max": 8.4}, "preferred_work_mode": "hybrid", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 110, "saved_by_recruiters_30d": 2, "interview_completion_rate": 0.32, "offer_acceptance_rate": 0.17, "verified_email": true, "verified_phone": false, "linkedin_connected": false}}, {"candidate_id": "CAND_0000018", "profile": {"anonymized_name": "Vivaan Reddy", "headline": "Frontend Engineer | Full-stack development", "summary": "Software engineer with 6.6 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. My background is full-stack, but my comfort zone is the backend and the database. I\'ve been keeping up with AI/ML at a self-learner level \\u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \\u2014 but I haven\'t done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "Bhubaneswar, Odisha", "country": "India", "years_of_experience": 6.6, "current_title": "Frontend Engineer", "current_company": "Acme Corp", "current_company_size": "201-500", "current_industry": "Manufacturing"}, "career_history": [{"company": "Acme Corp", "title": "Frontend Engineer", "start_date": "2023-09-10", "end_date": null, "duration_months": 33, "is_current": true, "industry": "Manufacturing", "company_size": "201-500", "description": "Test automation and QA engineering for a fintech product. Built and maintained the end-to-end test suite using Selenium and pytest, plus the load-testing setup using Locust. Worked closely with developers on testability patterns and with product on acceptance criteria. Recent work has been on shifting test responsibility into the dev team \\u2014 moving from QA-as-gate to QA-as-coach. Career has been entirely in QA/test engineering."}, {"company": "Pied Piper", "title": "Frontend Engineer", "start_date": "2021-01-23", "end_date": "2023-09-10", "duration_months": 32, "is_current": false, "industry": "Software", "company_size": "11-50", "description": "Test automation and QA engineering for a fintech product. Built and maintained the end-to-end test suite using Selenium and pytest, plus the load-testing setup using Locust. Worked closely with developers on testability patterns and with product on acceptance criteria. Recent work has been on shifting test responsibility into the dev team \\u2014 moving from QA-as-gate to QA-as-coach. Career has been entirely in QA/test engineering."}, {"company": "Initech", "title": "Full Stack Developer", "start_date": "2019-12-30", "end_date": "2021-01-23", "duration_months": 13, "is_current": false, "industry": "Software", "company_size": "51-200", "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."}], "education": [{"institution": "Lovely Professional University", "degree": "Ph.D", "field_of_study": "Computer Engineering", "start_year": 2016, "end_year": 2020, "grade": "7.25 CGPA", "tier": "tier_3"}], "skills": [{"name": "CNN", "proficiency": "advanced", "endorsements": 53, "duration_months": 55}, {"name": "Java", "proficiency": "intermediate", "endorsements": 10, "duration_months": 12}, {"name": "Accounting", "proficiency": "intermediate", "endorsements": 9, "duration_months": 20}, {"name": "Data Pipelines", "proficiency": "beginner", "endorsements": 3, "duration_months": 13}, {"name": "Node.js", "proficiency": "intermediate", "endorsements": 0, "duration_months": 9}, {"name": "Tailwind", "proficiency": "beginner", "endorsements": 10, "duration_months": 10}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "native"}], "redrob_signals": {"profile_completeness_score": 34.8, "signup_date": "2025-07-09", "last_active_date": "2026-02-18", "open_to_work_flag": false, "profile_views_received_30d": 88, "applications_submitted_30d": 11, "recruiter_response_rate": 0.16, "avg_response_time_hours": 154.6, "skill_assessment_scores": {}, "connection_count": 284, "endorsements_received": 49, "notice_period_days": 120, "expected_salary_range_inr_lpa": {"min": 12.3, "max": 26.4}, "preferred_work_mode": "onsite", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 41, "saved_by_recruiters_30d": 16, "interview_completion_rate": 0.7, "offer_acceptance_rate": 0.46, "verified_email": false, "verified_phone": false, "linkedin_connected": false}}, {"candidate_id": "CAND_0000019", "profile": {"anonymized_name": "Myra Mishra", "headline": "Project Manager | 6.5+ yrs experience", "summary": "Professional with 6.5+ years of experience. I\'ve spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Trivandrum, Kerala", "country": "India", "years_of_experience": 6.5, "current_title": "Project Manager", "current_company": "Wayne Enterprises", "current_company_size": "10001+", "current_industry": "Conglomerate"}, "career_history": [{"company": "Wayne Enterprises", "title": "Project Manager", "start_date": "2022-10-15", "end_date": null, "duration_months": 44, "is_current": true, "industry": "Conglomerate", "company_size": "10001+", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}, {"company": "Wayne Enterprises", "title": "Marketing Manager", "start_date": "2020-08-19", "end_date": "2022-10-08", "duration_months": 26, "is_current": false, "industry": "Conglomerate", "company_size": "10001+", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}, {"company": "Pied Piper", "title": "Business Analyst", "start_date": "2020-01-15", "end_date": "2020-08-12", "duration_months": 7, "is_current": false, "industry": "Software", "company_size": "11-50", "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."}], "education": [{"institution": "IISc Bangalore", "degree": "M.Tech", "field_of_study": "Computer Science", "start_year": 2010, "end_year": 2014, "grade": "72%", "tier": "tier_1"}, {"institution": "IIT Guwahati", "degree": "M.Tech", "field_of_study": "Machine Learning", "start_year": 2002, "end_year": 2006, "grade": "7.34 CGPA", "tier": "tier_1"}], "skills": [{"name": "Figma", "proficiency": "intermediate", "endorsements": 13, "duration_months": 34}, {"name": "GraphQL", "proficiency": "beginner", "endorsements": 4, "duration_months": 12}, {"name": "Six Sigma", "proficiency": "beginner", "endorsements": 4, "duration_months": 10}, {"name": "Scrum", "proficiency": "beginner", "endorsements": 15, "duration_months": 2}, {"name": "YOLO", "proficiency": "intermediate", "endorsements": 10, "duration_months": 34}, {"name": "gRPC", "proficiency": "beginner", "endorsements": 12, "duration_months": 9}, {"name": "AWS", "proficiency": "intermediate", "endorsements": 11, "duration_months": 28}, {"name": "Azure", "proficiency": "intermediate", "endorsements": 6, "duration_months": 21}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 38.6, "signup_date": "2025-07-20", "last_active_date": "2026-05-21", "open_to_work_flag": false, "profile_views_received_30d": 61, "applications_submitted_30d": 9, "recruiter_response_rate": 0.34, "avg_response_time_hours": 100.0, "skill_assessment_scores": {}, "connection_count": 593, "endorsements_received": 25, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 12.5, "max": 7.7}, "preferred_work_mode": "hybrid", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 141, "saved_by_recruiters_30d": 0, "interview_completion_rate": 0.31, "offer_acceptance_rate": -1, "verified_email": false, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000020", "profile": {"anonymized_name": "Aditya Iyengar", "headline": "Mechanical Engineer | 6.3+ yrs experience", "summary": "Professional with 6.3+ years of experience. I\'m a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Ahmedabad, Gujarat", "country": "India", "years_of_experience": 6.3, "current_title": "Mechanical Engineer", "current_company": "Wipro", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Wipro", "title": "Mechanical Engineer", "start_date": "2023-06-12", "end_date": null, "duration_months": 36, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \\u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}, {"company": "Stark Industries", "title": "Graphic Designer", "start_date": "2020-07-27", "end_date": "2023-04-13", "duration_months": 33, "is_current": false, "industry": "Manufacturing", "company_size": "1001-5000", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \\u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}, {"company": "Dunder Mifflin", "title": "Civil Engineer", "start_date": "2020-01-29", "end_date": "2020-07-27", "duration_months": 6, "is_current": false, "industry": "Paper Products", "company_size": "201-500", "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \\u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."}], "education": [{"institution": "IIT Kharagpur", "degree": "B.E.", "field_of_study": "Computer Science", "start_year": 2018, "end_year": 2022, "grade": "6.67 CGPA", "tier": "tier_1"}], "skills": [{"name": "GraphQL", "proficiency": "beginner", "endorsements": 15, "duration_months": 18}, {"name": "TypeScript", "proficiency": "beginner", "endorsements": 7, "duration_months": 8}, {"name": "Flask", "proficiency": "beginner", "endorsements": 11, "duration_months": 16}, {"name": "Weights & Biases", "proficiency": "advanced", "endorsements": 50, "duration_months": 30}, {"name": "GCP", "proficiency": "intermediate", "endorsements": 2, "duration_months": 34}, {"name": "Salesforce CRM", "proficiency": "beginner", "endorsements": 8, "duration_months": 16}, {"name": "HTML", "proficiency": "intermediate", "endorsements": 2, "duration_months": 35}], "certifications": [{"name": "Scrum Master Certified", "issuer": "Scrum Alliance", "year": 2021}], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 73.0, "signup_date": "2023-01-26", "last_active_date": "2025-10-05", "open_to_work_flag": false, "profile_views_received_30d": 38, "applications_submitted_30d": 9, "recruiter_response_rate": 0.55, "avg_response_time_hours": 207.8, "skill_assessment_scores": {"Weights & Biases": 53.7}, "connection_count": 479, "endorsements_received": 35, "notice_period_days": 30, "expected_salary_range_inr_lpa": {"min": 17.2, "max": 18.2}, "preferred_work_mode": "remote", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 31, "saved_by_recruiters_30d": 11, "interview_completion_rate": 0.71, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": false, "linkedin_connected": true}}, {"candidate_id": "CAND_0000021", "profile": {"anonymized_name": "Rahul Joshi", "headline": "Project Manager | AI enthusiast | Building with LLMs", "summary": "Project Manager with 14.5+ years of experience driving outcomes in my domain. I have built strong functional expertise in the typical responsibilities of the role, including team management, stakeholder communication, and project delivery. Recently I\'ve been excited about how AI and GenAI tools can augment my work. I\'ve been taking online courses on RAG and vector databases, experimenting with LangChain and the OpenAI API for side projects, and exploring how LLMs can streamline workflows in my current function. Open to roles that combine my existing domain experience with emerging AI technologies \\u2014 I think the most interesting opportunities are at this intersection. Looking for positions where I can contribute both my functional expertise and grow my AI capabilities.", "location": "Bhubaneswar, Odisha", "country": "India", "years_of_experience": 14.5, "current_title": "Project Manager", "current_company": "Wipro", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Wipro", "title": "Project Manager", "start_date": "2023-12-09", "end_date": null, "duration_months": 30, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."}, {"company": "Infosys", "title": "Marketing Manager", "start_date": "2021-02-22", "end_date": "2023-10-10", "duration_months": 32, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}, {"company": "Stark Industries", "title": "Sales Executive", "start_date": "2019-08-25", "end_date": "2021-02-15", "duration_months": 18, "is_current": false, "industry": "Manufacturing", "company_size": "1001-5000", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}, {"company": "Dunder Mifflin", "title": "Customer Support", "start_date": "2015-06-17", "end_date": "2019-08-25", "duration_months": 51, "is_current": false, "industry": "Paper Products", "company_size": "201-500", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}, {"company": "Wipro", "title": "Project Manager", "start_date": "2014-03-24", "end_date": "2015-06-17", "duration_months": 15, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}, {"company": "TCS", "title": "Customer Support", "start_date": "2011-12-05", "end_date": "2014-01-23", "duration_months": 26, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}], "education": [{"institution": "Tier-3 Engineering College", "degree": "B.Tech", "field_of_study": "Artificial Intelligence", "start_year": 2008, "end_year": 2011, "grade": "9.30 CGPA", "tier": "tier_4"}], "skills": [{"name": "Hadoop", "proficiency": "beginner", "endorsements": 10, "duration_months": 5}, {"name": "PostgreSQL", "proficiency": "beginner", "endorsements": 10, "duration_months": 4}, {"name": "Kafka", "proficiency": "beginner", "endorsements": 6, "duration_months": 6}, {"name": "Microservices", "proficiency": "intermediate", "endorsements": 0, "duration_months": 14}, {"name": "AWS", "proficiency": "intermediate", "endorsements": 11, "duration_months": 26}, {"name": "TypeScript", "proficiency": "beginner", "endorsements": 6, "duration_months": 6}, {"name": "ETL", "proficiency": "beginner", "endorsements": 11, "duration_months": 3}, {"name": "Spring Boot", "proficiency": "beginner", "endorsements": 1, "duration_months": 12}, {"name": "Recommendation Systems", "proficiency": "advanced", "endorsements": 3, "duration_months": 13}, {"name": "Fine-tuning LLMs", "proficiency": "advanced", "endorsements": 4, "duration_months": 4}, {"name": "Prompt Engineering", "proficiency": "advanced", "endorsements": 4, "duration_months": 5}, {"name": "LangChain", "proficiency": "intermediate", "endorsements": 1, "duration_months": 7}, {"name": "Pinecone", "proficiency": "intermediate", "endorsements": 4, "duration_months": 16}, {"name": "Vector Search", "proficiency": "intermediate", "endorsements": 3, "duration_months": 13}, {"name": "Embeddings", "proficiency": "advanced", "endorsements": 4, "duration_months": 18}, {"name": "FAISS", "proficiency": "intermediate", "endorsements": 2, "duration_months": 8}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 58.5, "signup_date": "2026-02-10", "last_active_date": "2025-11-21", "open_to_work_flag": false, "profile_views_received_30d": 1, "applications_submitted_30d": 8, "recruiter_response_rate": 0.49, "avg_response_time_hours": 98.7, "skill_assessment_scores": {}, "connection_count": 52, "endorsements_received": 3, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 10.9, "max": 24.4}, "preferred_work_mode": "hybrid", "willing_to_relocate": true, "github_activity_score": 6.4, "search_appearance_30d": 8, "saved_by_recruiters_30d": 3, "interview_completion_rate": 0.53, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": true}}, {"candidate_id": "CAND_0000022", "profile": {"anonymized_name": "Shaurya Chatterjee", "headline": "Mechanical Engineer | Driving business outcomes", "summary": "Professional with 1.1+ years of experience. I\'m a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Sydney", "country": "Australia", "years_of_experience": 1.1, "current_title": "Mechanical Engineer", "current_company": "Hooli", "current_company_size": "1001-5000", "current_industry": "Software"}, "career_history": [{"company": "Hooli", "title": "Mechanical Engineer", "start_date": "2025-05-02", "end_date": null, "duration_months": 13, "is_current": true, "industry": "Software", "company_size": "1001-5000", "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \\u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."}], "education": [{"institution": "VJTI Mumbai", "degree": "M.E.", "field_of_study": "Information Technology", "start_year": 2002, "end_year": 2006, "grade": "8.23 CGPA", "tier": "tier_2"}], "skills": [{"name": "OpenCV", "proficiency": "intermediate", "endorsements": 2, "duration_months": 11}, {"name": "Django", "proficiency": "intermediate", "endorsements": 0, "duration_months": 23}, {"name": "Terraform", "proficiency": "intermediate", "endorsements": 11, "duration_months": 14}, {"name": "Scrum", "proficiency": "intermediate", "endorsements": 7, "duration_months": 18}, {"name": "SQL", "proficiency": "intermediate", "endorsements": 3, "duration_months": 36}, {"name": "Java", "proficiency": "beginner", "endorsements": 11, "duration_months": 15}, {"name": "AWS", "proficiency": "intermediate", "endorsements": 5, "duration_months": 29}, {"name": "Six Sigma", "proficiency": "beginner", "endorsements": 1, "duration_months": 4}], "certifications": [{"name": "AWS Certified Cloud Practitioner", "issuer": "AWS", "year": 2024}], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 63.1, "signup_date": "2022-12-26", "last_active_date": "2026-05-03", "open_to_work_flag": true, "profile_views_received_30d": 39, "applications_submitted_30d": 2, "recruiter_response_rate": 0.27, "avg_response_time_hours": 30.2, "skill_assessment_scores": {}, "connection_count": 538, "endorsements_received": 21, "notice_period_days": 150, "expected_salary_range_inr_lpa": {"min": 12.3, "max": 8.5}, "preferred_work_mode": "remote", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 136, "saved_by_recruiters_30d": 5, "interview_completion_rate": 0.45, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": false, "linkedin_connected": false}}, {"candidate_id": "CAND_0000023", "profile": {"anonymized_name": "Kavya Nair", "headline": "Software Engineer | Cloud & DevOps", "summary": "Software engineer with 3.7 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. Most of my work is conventional backend engineering \\u2014 APIs, databases, queues, deployments. I\'ve been keeping up with AI/ML at a self-learner level \\u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \\u2014 but I haven\'t done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "New York", "country": "USA", "years_of_experience": 3.7, "current_title": "Software Engineer", "current_company": "Acme Corp", "current_company_size": "201-500", "current_industry": "Manufacturing"}, "career_history": [{"company": "Acme Corp", "title": "Software Engineer", "start_date": "2025-04-02", "end_date": null, "duration_months": 14, "is_current": true, "industry": "Manufacturing", "company_size": "201-500", "description": "Test automation and QA engineering for a fintech product. Built and maintained the end-to-end test suite using Selenium and pytest, plus the load-testing setup using Locust. Worked closely with developers on testability patterns and with product on acceptance criteria. Recent work has been on shifting test responsibility into the dev team \\u2014 moving from QA-as-gate to QA-as-coach. Career has been entirely in QA/test engineering."}, {"company": "Flipkart", "title": "Frontend Engineer", "start_date": "2022-10-15", "end_date": "2025-04-02", "duration_months": 30, "is_current": false, "industry": "E-commerce", "company_size": "10001+", "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."}], "education": [{"institution": "VJTI Mumbai", "degree": "B.E.", "field_of_study": "Data Science", "start_year": 2009, "end_year": 2013, "grade": "9.43 CGPA", "tier": "tier_2"}], "skills": [{"name": "BigQuery", "proficiency": "beginner", "endorsements": 8, "duration_months": 6}, {"name": "Marketing", "proficiency": "intermediate", "endorsements": 0, "duration_months": 15}, {"name": "Node.js", "proficiency": "intermediate", "endorsements": 3, "duration_months": 28}, {"name": "Django", "proficiency": "beginner", "endorsements": 13, "duration_months": 16}, {"name": "Salesforce CRM", "proficiency": "beginner", "endorsements": 9, "duration_months": 18}, {"name": "MongoDB", "proficiency": "beginner", "endorsements": 8, "duration_months": 10}, {"name": "ETL", "proficiency": "beginner", "endorsements": 12, "duration_months": 16}, {"name": "Redis", "proficiency": "beginner", "endorsements": 4, "duration_months": 10}, {"name": "Illustrator", "proficiency": "beginner", "endorsements": 1, "duration_months": 2}, {"name": "Rust", "proficiency": "intermediate", "endorsements": 2, "duration_months": 16}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "native"}], "redrob_signals": {"profile_completeness_score": 50.7, "signup_date": "2025-09-13", "last_active_date": "2026-04-06", "open_to_work_flag": false, "profile_views_received_30d": 10, "applications_submitted_30d": 2, "recruiter_response_rate": 0.57, "avg_response_time_hours": 64.8, "skill_assessment_scores": {}, "connection_count": 763, "endorsements_received": 39, "notice_period_days": 30, "expected_salary_range_inr_lpa": {"min": 17.4, "max": 20.5}, "preferred_work_mode": "flexible", "willing_to_relocate": false, "github_activity_score": 48.5, "search_appearance_30d": 239, "saved_by_recruiters_30d": 14, "interview_completion_rate": 0.9, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000024", "profile": {"anonymized_name": "Rajesh Arora", "headline": "HR Manager | 7.5+ yrs experience", "summary": "Professional with 7.5+ years of experience. I\'ve spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Trivandrum, Kerala", "country": "India", "years_of_experience": 7.5, "current_title": "HR Manager", "current_company": "TCS", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "TCS", "title": "HR Manager", "start_date": "2023-04-13", "end_date": null, "duration_months": 38, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \\u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}, {"company": "Infosys", "title": "Accountant", "start_date": "2018-12-05", "end_date": "2023-02-12", "duration_months": 51, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \\u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."}], "education": [{"institution": "Symbiosis International", "degree": "Ph.D", "field_of_study": "Computer Science", "start_year": 2008, "end_year": 2013, "grade": "7.65 CGPA", "tier": "tier_3"}], "skills": [{"name": "Figma", "proficiency": "beginner", "endorsements": 14, "duration_months": 15}, {"name": "Kubernetes", "proficiency": "beginner", "endorsements": 8, "duration_months": 17}, {"name": "Forecasting", "proficiency": "advanced", "endorsements": 43, "duration_months": 30}, {"name": "ETL", "proficiency": "intermediate", "endorsements": 11, "duration_months": 12}, {"name": "Node.js", "proficiency": "intermediate", "endorsements": 3, "duration_months": 15}, {"name": "Docker", "proficiency": "beginner", "endorsements": 5, "duration_months": 5}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "native"}], "redrob_signals": {"profile_completeness_score": 63.7, "signup_date": "2022-08-30", "last_active_date": "2026-01-20", "open_to_work_flag": false, "profile_views_received_30d": 71, "applications_submitted_30d": 4, "recruiter_response_rate": 0.78, "avg_response_time_hours": 238.2, "skill_assessment_scores": {"Forecasting": 65.1}, "connection_count": 445, "endorsements_received": 41, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 9.9, "max": 22.1}, "preferred_work_mode": "flexible", "willing_to_relocate": true, "github_activity_score": 46.3, "search_appearance_30d": 84, "saved_by_recruiters_30d": 7, "interview_completion_rate": 0.72, "offer_acceptance_rate": -1, "verified_email": false, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000025", "profile": {"anonymized_name": "Anika Kumar", "headline": "Frontend Engineer | Cloud & DevOps", "summary": "Software engineer with 7.3 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. I\'ve worked across web frontends, REST APIs, and cloud deployments; comfortable in most parts of a typical SaaS stack. I\'ve been keeping up with AI/ML at a self-learner level \\u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \\u2014 but I haven\'t done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "Vizag, Andhra Pradesh", "country": "India", "years_of_experience": 7.3, "current_title": "Frontend Engineer", "current_company": "Tech Mahindra", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Tech Mahindra", "title": "Frontend Engineer", "start_date": "2023-09-10", "end_date": null, "duration_months": 33, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."}, {"company": "Mindtree", "title": "Frontend Engineer", "start_date": "2019-09-17", "end_date": "2023-08-27", "duration_months": 48, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering."}, {"company": "Zomato", "title": "Frontend Engineer", "start_date": "2019-03-21", "end_date": "2019-09-17", "duration_months": 6, "is_current": false, "industry": "Food Delivery", "company_size": "5001-10000", "description": "Frontend engineering at a media company. React, TypeScript, and the typical surrounding tooling (Webpack, Jest, Cypress). Built the company\'s design system from scratch and led the migration from a legacy AngularJS app. Strong on the frontend craft \\u2014 accessibility, performance, animations \\u2014 but limited backend exposure."}], "education": [{"institution": "Regional Technical Institute", "degree": "Ph.D", "field_of_study": "Mechanical Engineering", "start_year": 2006, "end_year": 2010, "grade": "8.18 CGPA", "tier": "tier_4"}], "skills": [{"name": "JavaScript", "proficiency": "intermediate", "endorsements": 10, "duration_months": 26}, {"name": "Spark", "proficiency": "intermediate", "endorsements": 0, "duration_months": 22}, {"name": "GCP", "proficiency": "beginner", "endorsements": 7, "duration_months": 13}, {"name": "TypeScript", "proficiency": "beginner", "endorsements": 2, "duration_months": 17}, {"name": "LangChain", "proficiency": "advanced", "endorsements": 15, "duration_months": 34}, {"name": "Apache Flink", "proficiency": "intermediate", "endorsements": 2, "duration_months": 19}, {"name": "ETL", "proficiency": "beginner", "endorsements": 1, "duration_months": 18}, {"name": "Redis", "proficiency": "beginner", "endorsements": 0, "duration_months": 10}, {"name": "Data Pipelines", "proficiency": "intermediate", "endorsements": 9, "duration_months": 32}], "certifications": [{"name": "Six Sigma Green Belt", "issuer": "ASQ", "year": 2018}, {"name": "AWS Certified Cloud Practitioner", "issuer": "AWS", "year": 2025}], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 70.7, "signup_date": "2023-12-18", "last_active_date": "2026-03-30", "open_to_work_flag": true, "profile_views_received_30d": 107, "applications_submitted_30d": 11, "recruiter_response_rate": 0.74, "avg_response_time_hours": 128.0, "skill_assessment_scores": {"LangChain": 40.0}, "connection_count": 276, "endorsements_received": 52, "notice_period_days": 120, "expected_salary_range_inr_lpa": {"min": 18.8, "max": 30.7}, "preferred_work_mode": "hybrid", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 74, "saved_by_recruiters_30d": 9, "interview_completion_rate": 0.7, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": true}}, {"candidate_id": "CAND_0000026", "profile": {"anonymized_name": "Neha Rao", "headline": "Graphic Designer | 6.8+ yrs experience", "summary": "Professional with 6.8+ years of experience. My professional background is in marketing manager \\u2014 I\'ve built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Kochi, Kerala", "country": "India", "years_of_experience": 6.8, "current_title": "Graphic Designer", "current_company": "Initech", "current_company_size": "51-200", "current_industry": "Software"}, "career_history": [{"company": "Initech", "title": "Graphic Designer", "start_date": "2022-11-14", "end_date": null, "duration_months": 43, "is_current": true, "industry": "Software", "company_size": "51-200", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}, {"company": "Acme Corp", "title": "Accountant", "start_date": "2021-03-24", "end_date": "2022-11-14", "duration_months": 20, "is_current": false, "industry": "Manufacturing", "company_size": "201-500", "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \\u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."}, {"company": "Acme Corp", "title": "Project Manager", "start_date": "2019-10-31", "end_date": "2021-03-24", "duration_months": 17, "is_current": false, "industry": "Manufacturing", "company_size": "201-500", "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."}], "education": [{"institution": "Generic State University", "degree": "M.Sc", "field_of_study": "Statistics", "start_year": 2008, "end_year": 2012, "grade": "81%", "tier": "tier_4"}, {"institution": "Lovely Professional University", "degree": "B.E.", "field_of_study": "Machine Learning", "start_year": 2008, "end_year": 2013, "grade": "83%", "tier": "tier_3"}], "skills": [{"name": "Apache Beam", "proficiency": "intermediate", "endorsements": 4, "duration_months": 18}, {"name": "Kubeflow", "proficiency": "intermediate", "endorsements": 14, "duration_months": 27}, {"name": "Scrum", "proficiency": "beginner", "endorsements": 12, "duration_months": 8}, {"name": "ETL", "proficiency": "beginner", "endorsements": 15, "duration_months": 17}, {"name": "Django", "proficiency": "beginner", "endorsements": 6, "duration_months": 11}, {"name": "Docker", "proficiency": "beginner", "endorsements": 4, "duration_months": 9}, {"name": "Airflow", "proficiency": "intermediate", "endorsements": 7, "duration_months": 21}, {"name": "Kubernetes", "proficiency": "intermediate", "endorsements": 13, "duration_months": 22}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "native"}], "redrob_signals": {"profile_completeness_score": 80.3, "signup_date": "2023-12-08", "last_active_date": "2025-10-03", "open_to_work_flag": false, "profile_views_received_30d": 75, "applications_submitted_30d": 7, "recruiter_response_rate": 0.59, "avg_response_time_hours": 45.4, "skill_assessment_scores": {}, "connection_count": 321, "endorsements_received": 8, "notice_period_days": 30, "expected_salary_range_inr_lpa": {"min": 17.1, "max": 8.0}, "preferred_work_mode": "hybrid", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 154, "saved_by_recruiters_30d": 11, "interview_completion_rate": 0.49, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": false, "linkedin_connected": true}}, {"candidate_id": "CAND_0000027", "profile": {"anonymized_name": "Avni Pandey", "headline": "DevOps Engineer | Cloud & DevOps", "summary": "Software engineer with 3.9 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. I\'ve worked across web frontends, REST APIs, and cloud deployments; comfortable in most parts of a typical SaaS stack. I\'ve been keeping up with AI/ML at a self-learner level \\u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \\u2014 but I haven\'t done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "Kolkata, West Bengal", "country": "India", "years_of_experience": 3.9, "current_title": "DevOps Engineer", "current_company": "Infosys", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Infosys", "title": "DevOps Engineer", "start_date": "2023-06-12", "end_date": null, "duration_months": 36, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Java backend development at a large enterprise \\u2014 Spring Boot microservices, Kafka for inter-service messaging, Postgres + Redis for storage. Worked on the customer onboarding flow which involved orchestrating multiple downstream services. Solid on the Spring ecosystem, transaction handling, and the operational side of Java services. Looking to either go deeper on distributed systems or expand into modern application stacks."}, {"company": "Wipro", "title": "DevOps Engineer", "start_date": "2022-08-02", "end_date": "2023-05-29", "duration_months": 10, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Full-stack web application development at a SaaS company. Built React-based admin interfaces and the Node.js REST API backing them. Worked across the stack: frontend components, REST endpoint design, PostgreSQL schema, deployment via Docker/Kubernetes. Comfortable in most parts of a typical web stack though my comfort zone is the backend and database side. Recent learning has been on the testing and CI/CD discipline."}], "education": [{"institution": "IIT Bombay", "degree": "Ph.D", "field_of_study": "Information Technology", "start_year": 2006, "end_year": 2010, "grade": "6.55 CGPA", "tier": "tier_1"}, {"institution": "VJTI Mumbai", "degree": "B.E.", "field_of_study": "Artificial Intelligence", "start_year": 2017, "end_year": 2020, "grade": "9.11 CGPA", "tier": "tier_2"}], "skills": [{"name": "Docker", "proficiency": "intermediate", "endorsements": 1, "duration_months": 9}, {"name": "YOLO", "proficiency": "advanced", "endorsements": 31, "duration_months": 20}, {"name": "PEFT", "proficiency": "advanced", "endorsements": 39, "duration_months": 35}, {"name": "Webpack", "proficiency": "intermediate", "endorsements": 0, "duration_months": 33}, {"name": "Data Science", "proficiency": "advanced", "endorsements": 18, "duration_months": 24}, {"name": "AWS", "proficiency": "beginner", "endorsements": 4, "duration_months": 16}, {"name": "Java", "proficiency": "intermediate", "endorsements": 2, "duration_months": 15}, {"name": "Angular", "proficiency": "intermediate", "endorsements": 4, "duration_months": 25}, {"name": "Databricks", "proficiency": "intermediate", "endorsements": 3, "duration_months": 31}, {"name": "ETL", "proficiency": "intermediate", "endorsements": 5, "duration_months": 11}, {"name": "Marketing", "proficiency": "beginner", "endorsements": 15, "duration_months": 14}, {"name": "Information Retrieval", "proficiency": "intermediate", "endorsements": 5, "duration_months": 32}, {"name": "Weights & Biases", "proficiency": "advanced", "endorsements": 31, "duration_months": 44}, {"name": "Terraform", "proficiency": "intermediate", "endorsements": 12, "duration_months": 19}, {"name": "SAP", "proficiency": "intermediate", "endorsements": 8, "duration_months": 9}, {"name": "Illustrator", "proficiency": "beginner", "endorsements": 1, "duration_months": 11}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 31.0, "signup_date": "2023-03-07", "last_active_date": "2026-05-07", "open_to_work_flag": true, "profile_views_received_30d": 89, "applications_submitted_30d": 5, "recruiter_response_rate": 0.58, "avg_response_time_hours": 112.3, "skill_assessment_scores": {"YOLO": 60.2, "PEFT": 50.5, "Data Science": 35.1}, "connection_count": 282, "endorsements_received": 24, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 17.9, "max": 20.2}, "preferred_work_mode": "hybrid", "willing_to_relocate": false, "github_activity_score": 38.6, "search_appearance_30d": 136, "saved_by_recruiters_30d": 6, "interview_completion_rate": 0.61, "offer_acceptance_rate": 0.65, "verified_email": false, "verified_phone": true, "linkedin_connected": true}}, {"candidate_id": "CAND_0000028", "profile": {"anonymized_name": "Rohan Krishnan", "headline": "Operations Manager | Driving business outcomes", "summary": "Professional with 1.1+ years of experience. My professional background is in marketing manager \\u2014 I\'ve built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Dubai", "country": "UAE", "years_of_experience": 1.1, "current_title": "Operations Manager", "current_company": "Wipro", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Wipro", "title": "Operations Manager", "start_date": "2025-05-02", "end_date": null, "duration_months": 13, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."}], "education": [{"institution": "Symbiosis International", "degree": "M.Tech", "field_of_study": "Mathematics", "start_year": 2014, "end_year": 2017, "grade": "7.85 CGPA", "tier": "tier_3"}], "skills": [{"name": "Snowflake", "proficiency": "beginner", "endorsements": 6, "duration_months": 3}, {"name": "React", "proficiency": "beginner", "endorsements": 11, "duration_months": 7}, {"name": "JavaScript", "proficiency": "beginner", "endorsements": 6, "duration_months": 10}, {"name": "Tailwind", "proficiency": "beginner", "endorsements": 13, "duration_months": 15}, {"name": "REST APIs", "proficiency": "intermediate", "endorsements": 6, "duration_months": 21}, {"name": "Photoshop", "proficiency": "intermediate", "endorsements": 9, "duration_months": 30}, {"name": "Data Pipelines", "proficiency": "intermediate", "endorsements": 4, "duration_months": 23}, {"name": "Terraform", "proficiency": "intermediate", "endorsements": 9, "duration_months": 18}, {"name": "CNN", "proficiency": "intermediate", "endorsements": 8, "duration_months": 29}, {"name": "Content Writing", "proficiency": "beginner", "endorsements": 9, "duration_months": 18}], "certifications": [{"name": "AWS Certified Cloud Practitioner", "issuer": "AWS", "year": 2020}], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 51.2, "signup_date": "2025-09-23", "last_active_date": "2026-03-31", "open_to_work_flag": false, "profile_views_received_30d": 6, "applications_submitted_30d": 7, "recruiter_response_rate": 0.14, "avg_response_time_hours": 13.2, "skill_assessment_scores": {}, "connection_count": 524, "endorsements_received": 1, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 12.9, "max": 17.2}, "preferred_work_mode": "remote", "willing_to_relocate": true, "github_activity_score": -1, "search_appearance_30d": 68, "saved_by_recruiters_30d": 4, "interview_completion_rate": 0.86, "offer_acceptance_rate": 0.49, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000029", "profile": {"anonymized_name": "Priya Sethi", "headline": "Business Analyst | Driving business outcomes", "summary": "Professional with 7.2+ years of experience. I\'m a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Noida, Uttar Pradesh", "country": "India", "years_of_experience": 7.2, "current_title": "Business Analyst", "current_company": "Wipro", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Wipro", "title": "Business Analyst", "start_date": "2025-02-01", "end_date": null, "duration_months": 16, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \\u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}, {"company": "Globex Inc", "title": "Mechanical Engineer", "start_date": "2023-12-09", "end_date": "2025-02-01", "duration_months": 14, "is_current": false, "industry": "Manufacturing", "company_size": "501-1000", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}, {"company": "TCS", "title": "Civil Engineer", "start_date": "2019-05-27", "end_date": "2023-12-02", "duration_months": 55, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product. Built out the support knowledge base and the agent training program. Strong on the people-management side and the process side; lighter on technical depth beyond product expertise."}], "education": [{"institution": "Symbiosis International", "degree": "B.Tech", "field_of_study": "Artificial Intelligence", "start_year": 2007, "end_year": 2011, "grade": "6.59 CGPA", "tier": "tier_3"}], "skills": [{"name": "Node.js", "proficiency": "beginner", "endorsements": 9, "duration_months": 18}, {"name": "Scrum", "proficiency": "beginner", "endorsements": 2, "duration_months": 5}, {"name": "Tailwind", "proficiency": "intermediate", "endorsements": 5, "duration_months": 21}, {"name": "Hadoop", "proficiency": "beginner", "endorsements": 10, "duration_months": 4}, {"name": "Spring Boot", "proficiency": "intermediate", "endorsements": 1, "duration_months": 10}, {"name": "CI/CD", "proficiency": "beginner", "endorsements": 4, "duration_months": 18}, {"name": "gRPC", "proficiency": "beginner", "endorsements": 15, "duration_months": 17}, {"name": "Terraform", "proficiency": "beginner", "endorsements": 9, "duration_months": 10}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "native"}], "redrob_signals": {"profile_completeness_score": 40.5, "signup_date": "2025-06-17", "last_active_date": "2025-09-29", "open_to_work_flag": false, "profile_views_received_30d": 51, "applications_submitted_30d": 8, "recruiter_response_rate": 0.12, "avg_response_time_hours": 48.4, "skill_assessment_scores": {}, "connection_count": 297, "endorsements_received": 4, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 17.5, "max": 19.1}, "preferred_work_mode": "onsite", "willing_to_relocate": false, "github_activity_score": 42.5, "search_appearance_30d": 150, "saved_by_recruiters_30d": 8, "interview_completion_rate": 0.67, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": false, "linkedin_connected": true}}, {"candidate_id": "CAND_0000030", "profile": {"anonymized_name": "Ritu Pillai", "headline": "Marketing Manager | Driving business outcomes", "summary": "Professional with 10.0+ years of experience. I\'ve spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Kochi, Kerala", "country": "India", "years_of_experience": 10.0, "current_title": "Marketing Manager", "current_company": "Dunder Mifflin", "current_company_size": "201-500", "current_industry": "Paper Products"}, "career_history": [{"company": "Dunder Mifflin", "title": "Marketing Manager", "start_date": "2022-03-19", "end_date": null, "duration_months": 51, "is_current": true, "industry": "Paper Products", "company_size": "201-500", "description": "Senior accounting role at a mid-sized company \\u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."}, {"company": "Hooli", "title": "Sales Executive", "start_date": "2018-07-08", "end_date": "2022-03-19", "duration_months": 45, "is_current": false, "industry": "Software", "company_size": "1001-5000", "description": "Operations management role at a logistics company. Owned daily fulfillment operations across 3 warehouses, managing a team of 80 across receiving, picking, packing, and outbound. Built and tracked the operational KPIs (on-time fulfillment, accuracy, cost per order) and led the continuous improvement initiatives that drove a 22% productivity gain over 18 months."}, {"company": "Hooli", "title": "Content Writer", "start_date": "2016-08-17", "end_date": "2018-06-08", "duration_months": 22, "is_current": false, "industry": "Software", "company_size": "1001-5000", "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \\u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."}], "education": [{"institution": "Generic State University", "degree": "B.E.", "field_of_study": "Computer Engineering", "start_year": 2010, "end_year": 2014, "grade": "81%", "tier": "tier_4"}, {"institution": "Generic State University", "degree": "Ph.D", "field_of_study": "Artificial Intelligence", "start_year": 2017, "end_year": 2020, "grade": "7.98 CGPA", "tier": "tier_4"}], "skills": [{"name": "gRPC", "proficiency": "intermediate", "endorsements": 3, "duration_months": 36}, {"name": "Apache Beam", "proficiency": "beginner", "endorsements": 5, "duration_months": 6}, {"name": "GraphQL", "proficiency": "intermediate", "endorsements": 2, "duration_months": 22}, {"name": "Java", "proficiency": "intermediate", "endorsements": 14, "duration_months": 11}, {"name": "Spring Boot", "proficiency": "intermediate", "endorsements": 2, "duration_months": 22}, {"name": "Microservices", "proficiency": "beginner", "endorsements": 11, "duration_months": 5}, {"name": "Six Sigma", "proficiency": "beginner", "endorsements": 8, "duration_months": 8}, {"name": "Accounting", "proficiency": "intermediate", "endorsements": 3, "duration_months": 30}, {"name": "HTML", "proficiency": "intermediate", "endorsements": 12, "duration_months": 9}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 59.7, "signup_date": "2025-09-25", "last_active_date": "2025-10-27", "open_to_work_flag": false, "profile_views_received_30d": 58, "applications_submitted_30d": 0, "recruiter_response_rate": 0.66, "avg_response_time_hours": 131.1, "skill_assessment_scores": {}, "connection_count": 552, "endorsements_received": 45, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 14.7, "max": 14.2}, "preferred_work_mode": "flexible", "willing_to_relocate": false, "github_activity_score": 21.7, "search_appearance_30d": 54, "saved_by_recruiters_30d": 1, "interview_completion_rate": 0.73, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000031", "profile": {"anonymized_name": "Ela Singh", "headline": "Recommendation Systems Engineer | Search, Ranking & Retrieval", "summary": "Machine learning engineer with 6.0 years of experience building ML-powered features in production. Strong background in NLP, recommendation systems, and applied AI; comfortable across the ML stack from feature engineering through deployment. Recently, I led the team that migrated our keyword-search-based product to embedding-based retrieval. I\'ve learned that most retrieval problems are actually evaluation problems in disguise. My academic background is in CS/ML but my main learning has come from shipping real systems and seeing what holds up under production load. Open to senior IC roles in applied ML or AI engineering, ideally at product companies where I\'d own a meaningful piece of the ML stack.", "location": "Hyderabad, Telangana", "country": "India", "years_of_experience": 6.0, "current_title": "Recommendation Systems Engineer", "current_company": "Swiggy", "current_company_size": "5001-10000", "current_industry": "Food Delivery"}, "career_history": [{"company": "Swiggy", "title": "Recommendation Systems Engineer", "start_date": "2025-04-02", "end_date": null, "duration_months": 14, "is_current": true, "industry": "Food Delivery", "company_size": "5001-10000", "description": "Trained and shipped multiple ranking models for our product\'s discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item engagement history. Owned the offline-online correlation analysis that determined which offline metrics actually predicted A/B test outcomes. Worked closely with PMs to define the optimization target (click-through vs. dwell time vs. downstream conversion) \\u2014 that work was as important as the modeling itself."}, {"company": "Mad Street Den", "title": "Search Engineer", "start_date": "2023-10-10", "end_date": "2025-02-01", "duration_months": 16, "is_current": false, "industry": "AI/ML", "company_size": "201-500", "description": "Trained and shipped multiple ranking models for our product\'s discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item engagement history. Owned the offline-online correlation analysis that determined which offline metrics actually predicted A/B test outcomes. Worked closely with PMs to define the optimization target (click-through vs. dwell time vs. downstream conversion) \\u2014 that work was as important as the modeling itself."}, {"company": "Uber", "title": "NLP Engineer", "start_date": "2021-07-22", "end_date": "2023-10-10", "duration_months": 27, "is_current": false, "industry": "Transportation", "company_size": "10001+", "description": "Trained and shipped multiple ranking models for our product\'s discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item engagement history. Owned the offline-online correlation analysis that determined which offline metrics actually predicted A/B test outcomes. Worked closely with PMs to define the optimization target (click-through vs. dwell time vs. downstream conversion) \\u2014 that work was as important as the modeling itself."}, {"company": "Zomato", "title": "Applied ML Engineer", "start_date": "2020-06-27", "end_date": "2021-07-22", "duration_months": 13, "is_current": false, "industry": "Food Delivery", "company_size": "5001-10000", "description": "Owned the ranking layer for an e-commerce search product, evolving it from a hand-tuned scoring function to a learning-to-rank model over 9 months. Designed the relevance labeling pipeline (mix of click-through data and explicit human judgments), the feature pipeline, and the training/eval workflow. Most of the work was infrastructure and data quality \\u2014 the modeling part was almost the easy bit. Final model improved revenue-per-search by 12%."}], "education": [{"institution": "SRM University", "degree": "M.Tech", "field_of_study": "Computer Engineering", "start_year": 2002, "end_year": 2006, "grade": "9.16 CGPA", "tier": "tier_2"}], "skills": [{"name": "Go", "proficiency": "intermediate", "endorsements": 7, "duration_months": 19}, {"name": "MLflow", "proficiency": "advanced", "endorsements": 59, "duration_months": 21}, {"name": "FAISS", "proficiency": "advanced", "endorsements": 19, "duration_months": 35}, {"name": "Pinecone", "proficiency": "expert", "endorsements": 34, "duration_months": 88}, {"name": "Angular", "proficiency": "beginner", "endorsements": 4, "duration_months": 18}, {"name": "Image Classification", "proficiency": "advanced", "endorsements": 56, "duration_months": 28}, {"name": "Machine Learning", "proficiency": "advanced", "endorsements": 43, "duration_months": 23}, {"name": "Speech Recognition", "proficiency": "intermediate", "endorsements": 14, "duration_months": 24}, {"name": "BentoML", "proficiency": "intermediate", "endorsements": 6, "duration_months": 14}, {"name": "MLOps", "proficiency": "intermediate", "endorsements": 15, "duration_months": 36}, {"name": "Embeddings", "proficiency": "expert", "endorsements": 48, "duration_months": 60}, {"name": "Information Retrieval", "proficiency": "expert", "endorsements": 2, "duration_months": 84}, {"name": "Hugging Face Transformers", "proficiency": "expert", "endorsements": 18, "duration_months": 36}, {"name": "Feature Engineering", "proficiency": "advanced", "endorsements": 38, "duration_months": 26}, {"name": "Sentence Transformers", "proficiency": "expert", "endorsements": 16, "duration_months": 69}, {"name": "scikit-learn", "proficiency": "advanced", "endorsements": 41, "duration_months": 60}, {"name": "Marketing", "proficiency": "intermediate", "endorsements": 11, "duration_months": 36}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "native"}], "redrob_signals": {"profile_completeness_score": 83.4, "signup_date": "2026-01-28", "last_active_date": "2026-05-24", "open_to_work_flag": true, "profile_views_received_30d": 194, "applications_submitted_30d": 2, "recruiter_response_rate": 0.91, "avg_response_time_hours": 76.1, "skill_assessment_scores": {"MLflow": 75.1, "FAISS": 68.4, "Pinecone": 53.6, "Image Classification": 57.1}, "connection_count": 832, "endorsements_received": 177, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 27.3, "max": 60.2}, "preferred_work_mode": "flexible", "willing_to_relocate": true, "github_activity_score": 32.6, "search_appearance_30d": 778, "saved_by_recruiters_30d": 13, "interview_completion_rate": 0.6, "offer_acceptance_rate": 0.38, "verified_email": false, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000032", "profile": {"anonymized_name": "Pranav Agarwal", "headline": ".NET Developer | Full-stack development", "summary": "Software engineer with 8.1 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. Most of my work is conventional backend engineering \\u2014 APIs, databases, queues, deployments. I\'ve been keeping up with AI/ML at a self-learner level \\u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \\u2014 but I haven\'t done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "Gurgaon, Haryana", "country": "India", "years_of_experience": 8.1, "current_title": ".NET Developer", "current_company": "Cognizant", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Cognizant", "title": ".NET Developer", "start_date": "2024-02-07", "end_date": null, "duration_months": 28, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Java backend development at a large enterprise \\u2014 Spring Boot microservices, Kafka for inter-service messaging, Postgres + Redis for storage. Worked on the customer onboarding flow which involved orchestrating multiple downstream services. Solid on the Spring ecosystem, transaction handling, and the operational side of Java services. Looking to either go deeper on distributed systems or expand into modern application stacks."}, {"company": "HCL", "title": "Cloud Engineer", "start_date": "2021-11-19", "end_date": "2024-02-07", "duration_months": 27, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Test automation and QA engineering for a fintech product. Built and maintained the end-to-end test suite using Selenium and pytest, plus the load-testing setup using Locust. Worked closely with developers on testability patterns and with product on acceptance criteria. Recent work has been on shifting test responsibility into the dev team \\u2014 moving from QA-as-gate to QA-as-coach. Career has been entirely in QA/test engineering."}, {"company": "Globex Inc", "title": "Mobile Developer", "start_date": "2018-07-24", "end_date": "2021-11-05", "duration_months": 40, "is_current": false, "industry": "Manufacturing", "company_size": "501-1000", "description": "Full-stack web application development at a SaaS company. Built React-based admin interfaces and the Node.js REST API backing them. Worked across the stack: frontend components, REST endpoint design, PostgreSQL schema, deployment via Docker/Kubernetes. Comfortable in most parts of a typical web stack though my comfort zone is the backend and database side. Recent learning has been on the testing and CI/CD discipline."}], "education": [{"institution": "VIT Chennai", "degree": "M.Sc", "field_of_study": "Machine Learning", "start_year": 2017, "end_year": 2020, "grade": "8.37 CGPA", "tier": "tier_3"}, {"institution": "Amity University", "degree": "Ph.D", "field_of_study": "Physics", "start_year": 2011, "end_year": 2015, "grade": "7.95 CGPA", "tier": "tier_3"}], "skills": [{"name": "Speech Recognition", "proficiency": "advanced", "endorsements": 36, "duration_months": 19}, {"name": "Project Management", "proficiency": "beginner", "endorsements": 6, "duration_months": 17}, {"name": "REST APIs", "proficiency": "beginner", "endorsements": 13, "duration_months": 6}, {"name": "CSS", "proficiency": "intermediate", "endorsements": 15, "duration_months": 27}, {"name": "Embeddings", "proficiency": "advanced", "endorsements": 30, "duration_months": 30}, {"name": "Hadoop", "proficiency": "beginner", "endorsements": 0, "duration_months": 8}, {"name": "Spark", "proficiency": "intermediate", "endorsements": 14, "duration_months": 30}, {"name": "Python", "proficiency": "intermediate", "endorsements": 2, "duration_months": 13}, {"name": "Data Pipelines", "proficiency": "beginner", "endorsements": 6, "duration_months": 8}, {"name": "OpenCV", "proficiency": "advanced", "endorsements": 45, "duration_months": 54}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "native"}], "redrob_signals": {"profile_completeness_score": 35.4, "signup_date": "2023-12-20", "last_active_date": "2025-12-29", "open_to_work_flag": false, "profile_views_received_30d": 80, "applications_submitted_30d": 3, "recruiter_response_rate": 0.69, "avg_response_time_hours": 58.6, "skill_assessment_scores": {}, "connection_count": 404, "endorsements_received": 22, "notice_period_days": 150, "expected_salary_range_inr_lpa": {"min": 18.3, "max": 15.7}, "preferred_work_mode": "flexible", "willing_to_relocate": true, "github_activity_score": -1, "search_appearance_30d": 56, "saved_by_recruiters_30d": 4, "interview_completion_rate": 0.78, "offer_acceptance_rate": 0.25, "verified_email": true, "verified_phone": false, "linkedin_connected": false}}, {"candidate_id": "CAND_0000033", "profile": {"anonymized_name": "Shreya Nair", "headline": "Graphic Designer | Helping teams scale", "summary": "Professional with 8.6+ years of experience. I\'m a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Pune, Maharashtra", "country": "India", "years_of_experience": 8.6, "current_title": "Graphic Designer", "current_company": "Wipro", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Wipro", "title": "Graphic Designer", "start_date": "2023-11-09", "end_date": null, "duration_months": 31, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}, {"company": "Dunder Mifflin", "title": "Project Manager", "start_date": "2020-07-20", "end_date": "2023-11-02", "duration_months": 40, "is_current": false, "industry": "Paper Products", "company_size": "201-500", "description": "Senior accounting role at a mid-sized company \\u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."}, {"company": "Acme Corp", "title": "Content Writer", "start_date": "2018-01-02", "end_date": "2020-07-20", "duration_months": 31, "is_current": false, "industry": "Manufacturing", "company_size": "201-500", "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."}], "education": [{"institution": "Tier-3 Engineering College", "degree": "M.S.", "field_of_study": "Computer Science", "start_year": 2014, "end_year": 2017, "grade": "9.32 CGPA", "tier": "tier_4"}], "skills": [{"name": "Kubernetes", "proficiency": "beginner", "endorsements": 0, "duration_months": 9}, {"name": "Data Pipelines", "proficiency": "beginner", "endorsements": 4, "duration_months": 8}, {"name": "Snowflake", "proficiency": "intermediate", "endorsements": 11, "duration_months": 12}, {"name": "CI/CD", "proficiency": "intermediate", "endorsements": 11, "duration_months": 29}, {"name": "SEO", "proficiency": "intermediate", "endorsements": 9, "duration_months": 36}, {"name": "ETL", "proficiency": "intermediate", "endorsements": 0, "duration_months": 20}, {"name": "Airflow", "proficiency": "intermediate", "endorsements": 11, "duration_months": 16}, {"name": "TypeScript", "proficiency": "intermediate", "endorsements": 13, "duration_months": 15}, {"name": "Content Writing", "proficiency": "intermediate", "endorsements": 13, "duration_months": 11}, {"name": "Spring Boot", "proficiency": "intermediate", "endorsements": 5, "duration_months": 26}], "certifications": [{"name": "Six Sigma Green Belt", "issuer": "ASQ", "year": 2019}], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 74.0, "signup_date": "2026-03-13", "last_active_date": "2026-03-27", "open_to_work_flag": true, "profile_views_received_30d": 42, "applications_submitted_30d": 9, "recruiter_response_rate": 0.08, "avg_response_time_hours": 210.9, "skill_assessment_scores": {}, "connection_count": 410, "endorsements_received": 29, "notice_period_days": 30, "expected_salary_range_inr_lpa": {"min": 8.3, "max": 13.0}, "preferred_work_mode": "remote", "willing_to_relocate": false, "github_activity_score": 38.3, "search_appearance_30d": 98, "saved_by_recruiters_30d": 2, "interview_completion_rate": 0.37, "offer_acceptance_rate": -1, "verified_email": false, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000034", "profile": {"anonymized_name": "Yash Chatterjee", "headline": "Business Analyst | Driving business outcomes", "summary": "Professional with 14.5+ years of experience. I\'m a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Ahmedabad, Gujarat", "country": "India", "years_of_experience": 14.5, "current_title": "Business Analyst", "current_company": "Wipro", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "Wipro", "title": "Business Analyst", "start_date": "2025-04-02", "end_date": null, "duration_months": 14, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Content writing and SEO strategy for a tech-focused publication. Wrote longform articles on developer tools, cloud platforms, and AI/ML topics \\u2014 including some that ranked on the first page of search for high-competition keywords. Managed a freelance writer pool and the editorial calendar. Recent work has been on AI-assisted content production, using LLM tools for research, drafting, and editing while maintaining editorial quality."}, {"company": "Hooli", "title": "Mechanical Engineer", "start_date": "2023-05-13", "end_date": "2025-02-01", "duration_months": 21, "is_current": false, "industry": "Software", "company_size": "1001-5000", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}, {"company": "Infosys", "title": "Business Analyst", "start_date": "2021-02-22", "end_date": "2023-04-13", "duration_months": 26, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}, {"company": "TCS", "title": "Accountant", "start_date": "2017-11-10", "end_date": "2021-02-22", "duration_months": 40, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}, {"company": "Hooli", "title": "Accountant", "start_date": "2016-01-20", "end_date": "2017-11-10", "duration_months": 22, "is_current": false, "industry": "Software", "company_size": "1001-5000", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \\u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}, {"company": "Pied Piper", "title": "Content Writer", "start_date": "2012-12-06", "end_date": "2016-01-20", "duration_months": 38, "is_current": false, "industry": "Software", "company_size": "11-50", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \\u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}, {"company": "Stark Industries", "title": "Content Writer", "start_date": "2012-01-11", "end_date": "2012-10-07", "duration_months": 9, "is_current": false, "industry": "Manufacturing", "company_size": "1001-5000", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}], "education": [{"institution": "Tier-3 Engineering College", "degree": "B.E.", "field_of_study": "Computer Engineering", "start_year": 2005, "end_year": 2010, "grade": "8.97 CGPA", "tier": "tier_4"}], "skills": [{"name": "GraphQL", "proficiency": "beginner", "endorsements": 2, "duration_months": 3}, {"name": "Excel", "proficiency": "intermediate", "endorsements": 15, "duration_months": 19}, {"name": "Node.js", "proficiency": "beginner", "endorsements": 6, "duration_months": 6}, {"name": "Terraform", "proficiency": "intermediate", "endorsements": 6, "duration_months": 13}, {"name": "Salesforce CRM", "proficiency": "intermediate", "endorsements": 6, "duration_months": 27}, {"name": "Flask", "proficiency": "intermediate", "endorsements": 8, "duration_months": 28}, {"name": "React", "proficiency": "beginner", "endorsements": 14, "duration_months": 2}, {"name": "Azure", "proficiency": "beginner", "endorsements": 1, "duration_months": 5}, {"name": "Redux", "proficiency": "beginner", "endorsements": 15, "duration_months": 3}, {"name": "Next.js", "proficiency": "intermediate", "endorsements": 7, "duration_months": 21}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "native"}], "redrob_signals": {"profile_completeness_score": 41.2, "signup_date": "2024-01-15", "last_active_date": "2026-01-03", "open_to_work_flag": false, "profile_views_received_30d": 25, "applications_submitted_30d": 7, "recruiter_response_rate": 0.15, "avg_response_time_hours": 253.5, "skill_assessment_scores": {}, "connection_count": 226, "endorsements_received": 27, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 14.4, "max": 28.0}, "preferred_work_mode": "hybrid", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 143, "saved_by_recruiters_30d": 2, "interview_completion_rate": 0.41, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000035", "profile": {"anonymized_name": "Vikram Verma", "headline": "Full Stack Developer | Backend systems & APIs", "summary": "Software engineer with 4.3 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. My background is full-stack, but my comfort zone is the backend and the database. I\'ve been keeping up with AI/ML at a self-learner level \\u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \\u2014 but I haven\'t done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "Hyderabad, Telangana", "country": "India", "years_of_experience": 4.3, "current_title": "Full Stack Developer", "current_company": "Globex Inc", "current_company_size": "501-1000", "current_industry": "Manufacturing"}, "career_history": [{"company": "Globex Inc", "title": "Full Stack Developer", "start_date": "2023-09-10", "end_date": null, "duration_months": 33, "is_current": true, "industry": "Manufacturing", "company_size": "501-1000", "description": "Full-stack web application development at a SaaS company. Built React-based admin interfaces and the Node.js REST API backing them. Worked across the stack: frontend components, REST endpoint design, PostgreSQL schema, deployment via Docker/Kubernetes. Comfortable in most parts of a typical web stack though my comfort zone is the backend and database side. Recent learning has been on the testing and CI/CD discipline."}, {"company": "Wipro", "title": "Software Engineer", "start_date": "2022-03-19", "end_date": "2023-09-10", "duration_months": 18, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Frontend engineering at a media company. React, TypeScript, and the typical surrounding tooling (Webpack, Jest, Cypress). Built the company\'s design system from scratch and led the migration from a legacy AngularJS app. Strong on the frontend craft \\u2014 accessibility, performance, animations \\u2014 but limited backend exposure."}], "education": [{"institution": "Generic State University", "degree": "B.E.", "field_of_study": "Civil Engineering", "start_year": 2010, "end_year": 2013, "grade": "90%", "tier": "tier_4"}, {"institution": "Bharati Vidyapeeth", "degree": "M.S.", "field_of_study": "Data Science", "start_year": 2010, "end_year": 2015, "grade": "7.08 CGPA", "tier": "tier_3"}], "skills": [{"name": "Snowflake", "proficiency": "intermediate", "endorsements": 11, "duration_months": 27}, {"name": "BigQuery", "proficiency": "beginner", "endorsements": 15, "duration_months": 6}, {"name": "Recommendation Systems", "proficiency": "intermediate", "endorsements": 2, "duration_months": 34}, {"name": "Data Pipelines", "proficiency": "beginner", "endorsements": 14, "duration_months": 18}, {"name": "Docker", "proficiency": "beginner", "endorsements": 3, "duration_months": 11}, {"name": "MongoDB", "proficiency": "intermediate", "endorsements": 11, "duration_months": 16}, {"name": "PostgreSQL", "proficiency": "intermediate", "endorsements": 14, "duration_months": 13}, {"name": "Sales", "proficiency": "intermediate", "endorsements": 0, "duration_months": 19}, {"name": "Kafka", "proficiency": "intermediate", "endorsements": 14, "duration_months": 29}, {"name": "Speech Recognition", "proficiency": "intermediate", "endorsements": 5, "duration_months": 33}, {"name": "BentoML", "proficiency": "advanced", "endorsements": 40, "duration_months": 59}, {"name": "Go", "proficiency": "beginner", "endorsements": 1, "duration_months": 11}, {"name": "Next.js", "proficiency": "intermediate", "endorsements": 15, "duration_months": 22}, {"name": "dbt", "proficiency": "beginner", "endorsements": 2, "duration_months": 9}], "certifications": [{"name": "AWS Certified Cloud Practitioner", "issuer": "AWS", "year": 2022}, {"name": "Six Sigma Green Belt", "issuer": "ASQ", "year": 2020}], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 56.2, "signup_date": "2024-08-13", "last_active_date": "2026-02-06", "open_to_work_flag": false, "profile_views_received_30d": 7, "applications_submitted_30d": 4, "recruiter_response_rate": 0.34, "avg_response_time_hours": 178.0, "skill_assessment_scores": {}, "connection_count": 398, "endorsements_received": 45, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 11.2, "max": 22.8}, "preferred_work_mode": "onsite", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 173, "saved_by_recruiters_30d": 1, "interview_completion_rate": 0.5, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000036", "profile": {"anonymized_name": "Ananya Bose", "headline": "Project Manager | 11.3+ yrs experience", "summary": "Professional with 11.3+ years of experience. I\'m a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Trivandrum, Kerala", "country": "India", "years_of_experience": 11.3, "current_title": "Project Manager", "current_company": "Initech", "current_company_size": "51-200", "current_industry": "Software"}, "career_history": [{"company": "Initech", "title": "Project Manager", "start_date": "2025-02-01", "end_date": null, "duration_months": 16, "is_current": true, "industry": "Software", "company_size": "51-200", "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."}, {"company": "Hooli", "title": "Content Writer", "start_date": "2023-08-11", "end_date": "2025-02-01", "duration_months": 18, "is_current": false, "industry": "Software", "company_size": "1001-5000", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \\u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}, {"company": "Dunder Mifflin", "title": "HR Manager", "start_date": "2019-11-30", "end_date": "2023-08-11", "duration_months": 45, "is_current": false, "industry": "Paper Products", "company_size": "201-500", "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."}, {"company": "TCS", "title": "Civil Engineer", "start_date": "2017-12-10", "end_date": "2019-11-30", "duration_months": 24, "is_current": false, "industry": "IT Services", "company_size": "10001+", "description": "Senior accounting role at a mid-sized company \\u2014 month-end close, financial reporting, statutory compliance (GAAP / Ind-AS), and tax filings. Owned the GL, fixed-asset register, and the audit-readiness function. Managed a team of 3 staff accountants. Built strong process discipline around the close cycle, reducing close time from 12 days to 7 over the last two years."}, {"company": "Wayne Enterprises", "title": "Marketing Manager", "start_date": "2015-05-18", "end_date": "2017-12-03", "duration_months": 31, "is_current": false, "industry": "Conglomerate", "company_size": "10001+", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}], "education": [{"institution": "KIIT University", "degree": "M.S.", "field_of_study": "Commerce", "start_year": 2002, "end_year": 2006, "grade": "8.89 CGPA", "tier": "tier_3"}], "skills": [{"name": "Figma", "proficiency": "beginner", "endorsements": 6, "duration_months": 13}, {"name": "MongoDB", "proficiency": "beginner", "endorsements": 4, "duration_months": 8}, {"name": "PowerPoint", "proficiency": "beginner", "endorsements": 5, "duration_months": 3}, {"name": "CSS", "proficiency": "beginner", "endorsements": 5, "duration_months": 14}, {"name": "Excel", "proficiency": "intermediate", "endorsements": 9, "duration_months": 28}, {"name": "GraphQL", "proficiency": "beginner", "endorsements": 5, "duration_months": 18}, {"name": "Object Detection", "proficiency": "advanced", "endorsements": 39, "duration_months": 37}, {"name": "Vue.js", "proficiency": "beginner", "endorsements": 13, "duration_months": 4}, {"name": "Sales", "proficiency": "beginner", "endorsements": 4, "duration_months": 9}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 81.8, "signup_date": "2025-09-05", "last_active_date": "2025-12-12", "open_to_work_flag": true, "profile_views_received_30d": 70, "applications_submitted_30d": 4, "recruiter_response_rate": 0.4, "avg_response_time_hours": 236.6, "skill_assessment_scores": {}, "connection_count": 324, "endorsements_received": 3, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 15.3, "max": 9.7}, "preferred_work_mode": "remote", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 175, "saved_by_recruiters_30d": 0, "interview_completion_rate": 0.78, "offer_acceptance_rate": 0.46, "verified_email": true, "verified_phone": false, "linkedin_connected": false}}, {"candidate_id": "CAND_0000037", "profile": {"anonymized_name": "Ved Sen", "headline": "Business Analyst | 14.3+ yrs experience", "summary": "Professional with 14.3+ years of experience. I\'ve spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Dubai", "country": "UAE", "years_of_experience": 14.3, "current_title": "Business Analyst", "current_company": "Stark Industries", "current_company_size": "1001-5000", "current_industry": "Manufacturing"}, "career_history": [{"company": "Stark Industries", "title": "Business Analyst", "start_date": "2022-03-19", "end_date": null, "duration_months": 51, "is_current": true, "industry": "Manufacturing", "company_size": "1001-5000", "description": "Business analyst at a consulting firm, working primarily with retail and CPG clients. Conducted business diagnostics, process re-engineering work, and digital transformation strategy projects. Strong on stakeholder management, structured problem-solving, and the typical consulting toolkit (slide-craft, Excel modeling, executive communication). Recent project work involved AI-strategy advisory but my own technical depth in AI is limited."}, {"company": "Initech", "title": "Civil Engineer", "start_date": "2019-08-18", "end_date": "2022-03-05", "duration_months": 31, "is_current": false, "industry": "Software", "company_size": "51-200", "description": "Operations management role at a logistics company. Owned daily fulfillment operations across 3 warehouses, managing a team of 80 across receiving, picking, packing, and outbound. Built and tracked the operational KPIs (on-time fulfillment, accuracy, cost per order) and led the continuous improvement initiatives that drove a 22% productivity gain over 18 months."}, {"company": "Stark Industries", "title": "Content Writer", "start_date": "2016-01-06", "end_date": "2019-08-18", "duration_months": 44, "is_current": false, "industry": "Manufacturing", "company_size": "1001-5000", "description": "Operations management role at a logistics company. Owned daily fulfillment operations across 3 warehouses, managing a team of 80 across receiving, picking, packing, and outbound. Built and tracked the operational KPIs (on-time fulfillment, accuracy, cost per order) and led the continuous improvement initiatives that drove a 22% productivity gain over 18 months."}, {"company": "Hooli", "title": "Mechanical Engineer", "start_date": "2013-07-20", "end_date": "2016-01-06", "duration_months": 30, "is_current": false, "industry": "Software", "company_size": "1001-5000", "description": "Marketing leadership role at a B2B SaaS company. Owned the demand-generation function \\u2014 content marketing, paid acquisition, SEO, email nurture. Built and managed a team of 5 across content, performance marketing, and marketing operations. Worked closely with sales on lead-quality definitions and the SDR-handoff process. Recent focus has been on account-based marketing for our enterprise segment."}, {"company": "Acme Corp", "title": "HR Manager", "start_date": "2012-04-26", "end_date": "2013-06-20", "duration_months": 14, "is_current": false, "industry": "Manufacturing", "company_size": "201-500", "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."}], "education": [{"institution": "Tier-3 Engineering College", "degree": "B.Tech", "field_of_study": "Machine Learning", "start_year": 2001, "end_year": 2005, "grade": "69%", "tier": "tier_4"}, {"institution": "Lovely Professional University", "degree": "M.Tech", "field_of_study": "Statistics", "start_year": 2013, "end_year": 2016, "grade": "6.81 CGPA", "tier": "tier_3"}], "skills": [{"name": "Databricks", "proficiency": "intermediate", "endorsements": 12, "duration_months": 32}, {"name": "Docker", "proficiency": "intermediate", "endorsements": 6, "duration_months": 35}, {"name": "Flask", "proficiency": "beginner", "endorsements": 5, "duration_months": 14}, {"name": "AWS", "proficiency": "intermediate", "endorsements": 2, "duration_months": 33}, {"name": "Terraform", "proficiency": "beginner", "endorsements": 3, "duration_months": 18}, {"name": "Tally", "proficiency": "intermediate", "endorsements": 14, "duration_months": 24}, {"name": "TTS", "proficiency": "intermediate", "endorsements": 3, "duration_months": 13}, {"name": "Apache Beam", "proficiency": "intermediate", "endorsements": 15, "duration_months": 23}], "certifications": [{"name": "AWS Certified Cloud Practitioner", "issuer": "AWS", "year": 2024}], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 25.6, "signup_date": "2025-04-11", "last_active_date": "2025-12-11", "open_to_work_flag": false, "profile_views_received_30d": 80, "applications_submitted_30d": 6, "recruiter_response_rate": 0.78, "avg_response_time_hours": 107.1, "skill_assessment_scores": {}, "connection_count": 503, "endorsements_received": 13, "notice_period_days": 30, "expected_salary_range_inr_lpa": {"min": 8.8, "max": 14.2}, "preferred_work_mode": "onsite", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 178, "saved_by_recruiters_30d": 1, "interview_completion_rate": 0.82, "offer_acceptance_rate": 0.19, "verified_email": true, "verified_phone": true, "linkedin_connected": true}}, {"candidate_id": "CAND_0000038", "profile": {"anonymized_name": "Myra Trivedi", "headline": "Java Developer | Cloud & DevOps", "summary": "Software engineer with 6.7 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. My background is full-stack, but my comfort zone is the backend and the database. I\'ve been keeping up with AI/ML at a self-learner level \\u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \\u2014 but I haven\'t done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "Coimbatore, Tamil Nadu", "country": "India", "years_of_experience": 6.7, "current_title": "Java Developer", "current_company": "Swiggy", "current_company_size": "5001-10000", "current_industry": "Food Delivery"}, "career_history": [{"company": "Swiggy", "title": "Java Developer", "start_date": "2023-11-09", "end_date": null, "duration_months": 31, "is_current": true, "industry": "Food Delivery", "company_size": "5001-10000", "description": "Frontend engineering at a media company. React, TypeScript, and the typical surrounding tooling (Webpack, Jest, Cypress). Built the company\'s design system from scratch and led the migration from a legacy AngularJS app. Strong on the frontend craft \\u2014 accessibility, performance, animations \\u2014 but limited backend exposure."}, {"company": "Globex Inc", "title": ".NET Developer", "start_date": "2022-09-15", "end_date": "2023-11-09", "duration_months": 14, "is_current": false, "industry": "Manufacturing", "company_size": "501-1000", "description": "Java backend development at a large enterprise \\u2014 Spring Boot microservices, Kafka for inter-service messaging, Postgres + Redis for storage. Worked on the customer onboarding flow which involved orchestrating multiple downstream services. Solid on the Spring ecosystem, transaction handling, and the operational side of Java services. Looking to either go deeper on distributed systems or expand into modern application stacks."}, {"company": "Hooli", "title": "DevOps Engineer", "start_date": "2019-11-30", "end_date": "2022-09-15", "duration_months": 34, "is_current": false, "industry": "Software", "company_size": "1001-5000", "description": "Frontend engineering at a media company. React, TypeScript, and the typical surrounding tooling (Webpack, Jest, Cypress). Built the company\'s design system from scratch and led the migration from a legacy AngularJS app. Strong on the frontend craft \\u2014 accessibility, performance, animations \\u2014 but limited backend exposure."}], "education": [{"institution": "VIT Chennai", "degree": "B.Sc", "field_of_study": "Computer Engineering", "start_year": 2015, "end_year": 2020, "grade": "70%", "tier": "tier_3"}], "skills": [{"name": "Kubeflow", "proficiency": "intermediate", "endorsements": 3, "duration_months": 26}, {"name": "Django", "proficiency": "beginner", "endorsements": 2, "duration_months": 18}, {"name": "Redux", "proficiency": "intermediate", "endorsements": 3, "duration_months": 13}, {"name": "Weaviate", "proficiency": "advanced", "endorsements": 37, "duration_months": 27}, {"name": "PowerPoint", "proficiency": "beginner", "endorsements": 15, "duration_months": 9}, {"name": "Figma", "proficiency": "beginner", "endorsements": 9, "duration_months": 8}, {"name": "Docker", "proficiency": "beginner", "endorsements": 12, "duration_months": 3}, {"name": "GraphQL", "proficiency": "intermediate", "endorsements": 13, "duration_months": 27}, {"name": "Agile", "proficiency": "intermediate", "endorsements": 14, "duration_months": 24}, {"name": "MLOps", "proficiency": "intermediate", "endorsements": 13, "duration_months": 26}, {"name": "Azure", "proficiency": "intermediate", "endorsements": 14, "duration_months": 27}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 35.8, "signup_date": "2026-03-25", "last_active_date": "2026-03-31", "open_to_work_flag": true, "profile_views_received_30d": 102, "applications_submitted_30d": 9, "recruiter_response_rate": 0.2, "avg_response_time_hours": 61.0, "skill_assessment_scores": {}, "connection_count": 316, "endorsements_received": 51, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 9.2, "max": 15.9}, "preferred_work_mode": "flexible", "willing_to_relocate": true, "github_activity_score": 37.8, "search_appearance_30d": 300, "saved_by_recruiters_30d": 18, "interview_completion_rate": 0.75, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0000039", "profile": {"anonymized_name": "Sai Saxena", "headline": "Marketing Manager | Helping teams scale", "summary": "Professional with 3.9+ years of experience. My professional background is in marketing manager \\u2014 I\'ve built and led teams, owned KPIs, and driven business outcomes in this domain. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Bhubaneswar, Odisha", "country": "India", "years_of_experience": 3.9, "current_title": "Marketing Manager", "current_company": "Acme Corp", "current_company_size": "201-500", "current_industry": "Manufacturing"}, "career_history": [{"company": "Acme Corp", "title": "Marketing Manager", "start_date": "2024-08-05", "end_date": null, "duration_months": 22, "is_current": true, "industry": "Manufacturing", "company_size": "201-500", "description": "Mechanical engineering design role at a hardware-product company. Led the design of two product subsystems through full lifecycle: concept, DFM/DFMA review, prototype, production tooling. Comfortable with CAD (SolidWorks, Creo), FEA (ANSYS), and the typical hardware-development cadence. Worked closely with manufacturing partners on production scale-up."}, {"company": "Stark Industries", "title": "Mechanical Engineer", "start_date": "2022-08-16", "end_date": "2024-08-05", "duration_months": 24, "is_current": false, "industry": "Manufacturing", "company_size": "1001-5000", "description": "Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design."}], "education": [{"institution": "Chandigarh University", "degree": "B.Tech", "field_of_study": "Electrical Engineering", "start_year": 2009, "end_year": 2014, "grade": "66%", "tier": "tier_3"}, {"institution": "Chandigarh University", "degree": "M.E.", "field_of_study": "Artificial Intelligence", "start_year": 2007, "end_year": 2011, "grade": "82%", "tier": "tier_3"}], "skills": [{"name": "Spark", "proficiency": "intermediate", "endorsements": 9, "duration_months": 15}, {"name": "Tailwind", "proficiency": "intermediate", "endorsements": 9, "duration_months": 8}, {"name": "Sales", "proficiency": "beginner", "endorsements": 6, "duration_months": 8}, {"name": "CI/CD", "proficiency": "intermediate", "endorsements": 11, "duration_months": 35}, {"name": "Illustrator", "proficiency": "intermediate", "endorsements": 9, "duration_months": 21}, {"name": "Hadoop", "proficiency": "intermediate", "endorsements": 15, "duration_months": 26}, {"name": "Microservices", "proficiency": "intermediate", "endorsements": 13, "duration_months": 19}, {"name": "REST APIs", "proficiency": "beginner", "endorsements": 15, "duration_months": 12}, {"name": "AWS", "proficiency": "intermediate", "endorsements": 7, "duration_months": 18}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 28.3, "signup_date": "2025-04-14", "last_active_date": "2026-04-18", "open_to_work_flag": false, "profile_views_received_30d": 22, "applications_submitted_30d": 0, "recruiter_response_rate": 0.52, "avg_response_time_hours": 40.1, "skill_assessment_scores": {}, "connection_count": 102, "endorsements_received": 9, "notice_period_days": 30, "expected_salary_range_inr_lpa": {"min": 17.4, "max": 16.0}, "preferred_work_mode": "hybrid", "willing_to_relocate": true, "github_activity_score": -1, "search_appearance_30d": 146, "saved_by_recruiters_30d": 6, "interview_completion_rate": 0.74, "offer_acceptance_rate": -1, "verified_email": false, "verified_phone": true, "linkedin_connected": true}}, {"candidate_id": "CAND_0000040", "profile": {"anonymized_name": "Dev Vora", "headline": "Customer Support | Helping teams scale", "summary": "Professional with 1.6+ years of experience. I\'ve spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I\'ve been curious about how AI tools could augment my work \\u2014 I\'ve experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities.", "location": "Kochi, Kerala", "country": "India", "years_of_experience": 1.6, "current_title": "Customer Support", "current_company": "Globex Inc", "current_company_size": "501-1000", "current_industry": "Manufacturing"}, "career_history": [{"company": "Globex Inc", "title": "Customer Support", "start_date": "2024-11-03", "end_date": null, "duration_months": 19, "is_current": true, "industry": "Manufacturing", "company_size": "501-1000", "description": "Enterprise sales of cloud software solutions into the mid-market segment. Carried a $1.8M ARR quota and consistently delivered against it across the last three years. Owned the full sales cycle: prospecting, discovery, technical evaluation (with SE support), commercial negotiation, and close. Strong on consultative selling for technical buyers; comfortable engaging with both engineering and finance stakeholders."}], "education": [{"institution": "Local Engineering College", "degree": "B.Tech", "field_of_study": "MBA", "start_year": 2010, "end_year": 2013, "grade": "7.45 CGPA", "tier": "tier_4"}], "skills": [{"name": "SQL", "proficiency": "beginner", "endorsements": 12, "duration_months": 4}, {"name": "Spring Boot", "proficiency": "intermediate", "endorsements": 15, "duration_months": 27}, {"name": "Accounting", "proficiency": "intermediate", "endorsements": 15, "duration_months": 16}, {"name": "Rust", "proficiency": "beginner", "endorsements": 12, "duration_months": 15}, {"name": "Redux", "proficiency": "intermediate", "endorsements": 2, "duration_months": 19}, {"name": "SAP", "proficiency": "intermediate", "endorsements": 14, "duration_months": 25}, {"name": "Weights & Biases", "proficiency": "intermediate", "endorsements": 10, "duration_months": 21}, {"name": "REST APIs", "proficiency": "intermediate", "endorsements": 6, "duration_months": 18}, {"name": "Spark", "proficiency": "beginner", "endorsements": 4, "duration_months": 14}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 39.5, "signup_date": "2024-10-03", "last_active_date": "2026-01-31", "open_to_work_flag": false, "profile_views_received_30d": 52, "applications_submitted_30d": 6, "recruiter_response_rate": 0.46, "avg_response_time_hours": 268.3, "skill_assessment_scores": {}, "connection_count": 176, "endorsements_received": 35, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 7.6, "max": 8.2}, "preferred_work_mode": "remote", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 95, "saved_by_recruiters_30d": 1, "interview_completion_rate": 0.8, "offer_acceptance_rate": 0.44, "verified_email": true, "verified_phone": false, "linkedin_connected": false}}, {"candidate_id": "CAND_0007009", "profile": {"anonymized_name": "Anika Pillai", "headline": "Recommendation Systems Engineer | ML, NLP, Recommendation Systems", "summary": "Machine learning engineer with 7.9 years of experience building ML-powered features in production. Strong background in NLP, recommendation systems, and applied AI; comfortable across the ML stack from feature engineering through deployment. Recently, I shipped our first RAG-based feature this year and now own the eval framework for it. I care a lot about evaluation rigor \\u2014 too many teams ship models without offline benchmarks they trust. My academic background is in CS/ML but my main learning has come from shipping real systems and seeing what holds up under production load. Open to senior IC roles in applied ML or AI engineering, ideally at product companies where I\'d own a meaningful piece of the ML stack.", "location": "Noida, Uttar Pradesh", "country": "India", "years_of_experience": 7.9, "current_title": "Recommendation Systems Engineer", "current_company": "Wysa", "current_company_size": "51-200", "current_industry": "HealthTech AI"}, "career_history": [{"company": "Wysa", "title": "Recommendation Systems Engineer", "start_date": "2025-04-02", "end_date": null, "duration_months": 14, "is_current": true, "industry": "HealthTech AI", "company_size": "51-200", "description": "Implemented a RAG-based customer support chatbot integrated with our existing ticketing system. Built the document ingestion pipeline (chunking, embedding via OpenAI embeddings, storing in Pinecone) and the answer-generation layer (initially GPT-4, then a fine-tuned smaller model for cost control). Designed the evaluation framework with both automatic metrics (BLEU, ROUGE) and human-in-the-loop quality scores. Deployment cut average ticket resolution time by 31% for the supported categories."}, {"company": "PolicyBazaar", "title": "Recommendation Systems Engineer", "start_date": "2021-04-23", "end_date": "2025-04-02", "duration_months": 48, "is_current": false, "industry": "Insurance Tech", "company_size": "5001-10000", "description": "Implemented a RAG-based customer support chatbot integrated with our existing ticketing system. Built the document ingestion pipeline (chunking, embedding via OpenAI embeddings, storing in Pinecone) and the answer-generation layer (initially GPT-4, then a fine-tuned smaller model for cost control). Designed the evaluation framework with both automatic metrics (BLEU, ROUGE) and human-in-the-loop quality scores. Deployment cut average ticket resolution time by 31% for the supported categories."}, {"company": "Netflix", "title": "Recommendation Systems Engineer", "start_date": "2018-08-23", "end_date": "2021-04-09", "duration_months": 32, "is_current": false, "industry": "Media", "company_size": "10001+", "description": "Developed a semantic search feature for an internal knowledge base of ~500K documents. Used sentence-transformers (all-MiniLM-L6-v2 initially, later upgraded to bge-base) with FAISS for fast nearest-neighbor retrieval. Designed the query expansion module that handles vocabulary mismatch between user queries and document terms. Reported search-relevance improvement of 35% over the prior Elasticsearch BM25 setup, validated through human relevance judgments."}], "education": [{"institution": "Symbiosis International", "degree": "M.Tech", "field_of_study": "Data Science", "start_year": 2011, "end_year": 2014, "grade": "7.52 CGPA", "tier": "tier_3"}, {"institution": "Anna University", "degree": "M.Sc", "field_of_study": "Computer Science", "start_year": 2017, "end_year": 2021, "grade": "7.74 CGPA", "tier": "tier_2"}], "skills": [{"name": "Statistical Modeling", "proficiency": "advanced", "endorsements": 54, "duration_months": 37}, {"name": "ETL", "proficiency": "intermediate", "endorsements": 11, "duration_months": 32}, {"name": "Weaviate", "proficiency": "expert", "endorsements": 6, "duration_months": 40}, {"name": "Python", "proficiency": "expert", "endorsements": 38, "duration_months": 95}, {"name": "Embeddings", "proficiency": "expert", "endorsements": 28, "duration_months": 44}, {"name": "Learning to Rank", "proficiency": "expert", "endorsements": 56, "duration_months": 75}, {"name": "PEFT", "proficiency": "expert", "endorsements": 21, "duration_months": 69}, {"name": "BM25", "proficiency": "expert", "endorsements": 45, "duration_months": 71}, {"name": "LoRA", "proficiency": "expert", "endorsements": 17, "duration_months": 42}, {"name": "CNN", "proficiency": "advanced", "endorsements": 10, "duration_months": 54}, {"name": "Forecasting", "proficiency": "advanced", "endorsements": 18, "duration_months": 21}, {"name": "LLMs", "proficiency": "advanced", "endorsements": 46, "duration_months": 43}, {"name": "Sentence Transformers", "proficiency": "advanced", "endorsements": 16, "duration_months": 42}, {"name": "Weights & Biases", "proficiency": "advanced", "endorsements": 18, "duration_months": 31}, {"name": "Haystack", "proficiency": "expert", "endorsements": 3, "duration_months": 69}, {"name": "Data Science", "proficiency": "advanced", "endorsements": 34, "duration_months": 35}, {"name": "QLoRA", "proficiency": "expert", "endorsements": 58, "duration_months": 58}, {"name": "OpenSearch", "proficiency": "expert", "endorsements": 23, "duration_months": 96}], "certifications": [{"name": "AWS Certified Machine Learning Specialty", "issuer": "AWS", "year": 2022}, {"name": "LangChain for LLM Application Development", "issuer": "DeepLearning.AI", "year": 2025}], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "native"}], "redrob_signals": {"profile_completeness_score": 82.8, "signup_date": "2024-03-18", "last_active_date": "2026-03-19", "open_to_work_flag": true, "profile_views_received_30d": 85, "applications_submitted_30d": 13, "recruiter_response_rate": 0.62, "avg_response_time_hours": 41.8, "skill_assessment_scores": {"Statistical Modeling": 51.2}, "connection_count": 251, "endorsements_received": 120, "notice_period_days": 30, "expected_salary_range_inr_lpa": {"min": 24.7, "max": 49.6}, "preferred_work_mode": "remote", "willing_to_relocate": false, "github_activity_score": 69.2, "search_appearance_30d": 615, "saved_by_recruiters_30d": 5, "interview_completion_rate": 0.87, "offer_acceptance_rate": 0.49, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0016000", "profile": {"anonymized_name": "Aarav Bansal", "headline": "Full Stack Developer | Backend systems & APIs", "summary": "Software engineer with 2.0 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. I\'ve worked across web frontends, REST APIs, and cloud deployments; comfortable in most parts of a typical SaaS stack. I\'ve been keeping up with AI/ML at a self-learner level \\u2014 taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project \\u2014 but I haven\'t done it in a professional capacity yet. Open to roles where I can either deepen my software engineering work or, if the team is open to it, start contributing to ML-adjacent systems.", "location": "Sydney", "country": "Australia", "years_of_experience": 2.0, "current_title": "Full Stack Developer", "current_company": "Initech", "current_company_size": "51-200", "current_industry": "Software"}, "career_history": [{"company": "Initech", "title": "Full Stack Developer", "start_date": "2024-06-06", "end_date": null, "duration_months": 24, "is_current": true, "industry": "Software", "company_size": "51-200", "description": "Full-stack web application development at a SaaS company. Built React-based admin interfaces and the Node.js REST API backing them. Worked across the stack: frontend components, REST endpoint design, PostgreSQL schema, deployment via Docker/Kubernetes. Comfortable in most parts of a typical web stack though my comfort zone is the backend and database side. Recent learning has been on the testing and CI/CD discipline."}], "education": [{"institution": "Lovely Professional University", "degree": "B.Tech", "field_of_study": "Machine Learning", "start_year": 2002, "end_year": 2005, "grade": "8.67 CGPA", "tier": "tier_3"}, {"institution": "Christ University", "degree": "M.Tech", "field_of_study": "Electrical Engineering", "start_year": 2013, "end_year": 2016, "grade": "7.30 CGPA", "tier": "tier_3"}], "skills": [{"name": "Flask", "proficiency": "beginner", "endorsements": 0, "duration_months": 16}, {"name": "Spring Boot", "proficiency": "beginner", "endorsements": 15, "duration_months": 8}, {"name": "TypeScript", "proficiency": "expert", "endorsements": 0, "duration_months": 0}, {"name": "Go", "proficiency": "expert", "endorsements": 1, "duration_months": 0}, {"name": "REST APIs", "proficiency": "beginner", "endorsements": 10, "duration_months": 9}, {"name": "Docker", "proficiency": "expert", "endorsements": 3, "duration_months": 0}, {"name": "Terraform", "proficiency": "intermediate", "endorsements": 12, "duration_months": 12}, {"name": "Hadoop", "proficiency": "expert", "endorsements": 2, "duration_months": 0}, {"name": "AWS", "proficiency": "beginner", "endorsements": 8, "duration_months": 12}, {"name": "Photoshop", "proficiency": "expert", "endorsements": 1, "duration_months": 0}], "certifications": [{"name": "Six Sigma Green Belt", "issuer": "ASQ", "year": 2022}, {"name": "AWS Certified Cloud Practitioner", "issuer": "AWS", "year": 2018}], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "native"}], "redrob_signals": {"profile_completeness_score": 35.8, "signup_date": "2024-12-11", "last_active_date": "2026-03-22", "open_to_work_flag": true, "profile_views_received_30d": 35, "applications_submitted_30d": 11, "recruiter_response_rate": 0.55, "avg_response_time_hours": 18.1, "skill_assessment_scores": {}, "connection_count": 754, "endorsements_received": 20, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 9.5, "max": 31.4}, "preferred_work_mode": "remote", "willing_to_relocate": false, "github_activity_score": -1, "search_appearance_30d": 154, "saved_by_recruiters_30d": 0, "interview_completion_rate": 0.92, "offer_acceptance_rate": 0.49, "verified_email": true, "verified_phone": false, "linkedin_connected": true}}, {"candidate_id": "CAND_0018499", "profile": {"anonymized_name": "Aarav Trivedi", "headline": "Senior Machine Learning Engineer | Building AI-native search & ranking systems", "summary": "Senior AI engineer with 7.2 years of hands-on experience building production ML systems, with a focus on search, retrieval, and ranking. Most recently, I designed the company\'s first hybrid retrieval system combining BM25 with dense vector recall, serving 50M+ queries per month. My day-to-day work spans embedding model selection and fine-tuning, hybrid retrieval architecture, learning-to-rank, behavioral-signal integration, and the offline/online evaluation that ties it all together. I\'ve shipped systems in both early-stage product companies and at larger scale, and I\'ve spent enough time on both that I know which tradeoffs apply where. I have strong opinions about when LLMs are the right hammer and when classical IR is \\u2014 usually it\'s both. Currently exploring my next move \\u2014 looking for senior IC or tech-lead roles where I can own the intelligence layer end-to-end.", "location": "Noida, Uttar Pradesh", "country": "India", "years_of_experience": 7.2, "current_title": "Senior Machine Learning Engineer", "current_company": "Zomato", "current_company_size": "5001-10000", "current_industry": "Food Delivery"}, "career_history": [{"company": "Zomato", "title": "Senior Machine Learning Engineer", "start_date": "2024-04-07", "end_date": null, "duration_months": 26, "is_current": true, "industry": "Food Delivery", "company_size": "5001-10000", "description": "Built a RAG-based ranking pipeline serving 50M+ queries per month for an internal recruiter-facing search product. The architecture combined BM25 + dense retrieval (BGE embeddings, FAISS HNSW) with an LLM-based re-ranker on the top-50, falling back to a learning-to-rank model when latency budget was tight. Designed the offline evaluation framework from scratch \\u2014 NDCG, MRR, recall@K calibrated against online A/B engagement metrics. Drove the migration over 4 months including the recruiter-feedback loop that surfaced reranking edge cases."}, {"company": "Google", "title": "Staff Machine Learning Engineer", "start_date": "2022-10-15", "end_date": "2024-04-07", "duration_months": 18, "is_current": false, "industry": "Internet", "company_size": "10001+", "description": "Built a RAG-based ranking pipeline serving 50M+ queries per month for an internal recruiter-facing search product. The architecture combined BM25 + dense retrieval (BGE embeddings, FAISS HNSW) with an LLM-based re-ranker on the top-50, falling back to a learning-to-rank model when latency budget was tight. Designed the offline evaluation framework from scratch \\u2014 NDCG, MRR, recall@K calibrated against online A/B engagement metrics. Drove the migration over 4 months including the recruiter-feedback loop that surfaced reranking edge cases."}, {"company": "Flipkart", "title": "Senior Machine Learning Engineer", "start_date": "2019-04-27", "end_date": "2022-10-08", "duration_months": 42, "is_current": false, "industry": "E-commerce", "company_size": "10001+", "description": "Fine-tuned LLaMA-2-7B and Mistral-7B variants using LoRA and QLoRA for domain-specific candidate-JD matching. Built the data curation pipeline that generated 200K high-quality preference pairs from recruiter labels, plus the eval harness using both ranking metrics and human-quality scores. Deployed the model via BentoML on Kubernetes with sub-200ms p95 latency by quantizing to INT8 and batching at the request level. Cost per inference dropped from $0.04 with GPT-3.5-fallback to under $0.001."}], "education": [{"institution": "Massachusetts Institute of Technology", "degree": "B.Sc", "field_of_study": "Artificial Intelligence", "start_year": 2013, "end_year": 2017, "grade": "6.54 CGPA", "tier": "tier_1"}, {"institution": "NIT Surathkal", "degree": "M.S.", "field_of_study": "Data Science", "start_year": 2017, "end_year": 2021, "grade": "69%", "tier": "tier_1"}], "skills": [{"name": "Deep Learning", "proficiency": "expert", "endorsements": 46, "duration_months": 53}, {"name": "Weaviate", "proficiency": "expert", "endorsements": 6, "duration_months": 88}, {"name": "Recommendation Systems", "proficiency": "expert", "endorsements": 52, "duration_months": 46}, {"name": "scikit-learn", "proficiency": "expert", "endorsements": 58, "duration_months": 93}, {"name": "Diffusion Models", "proficiency": "advanced", "endorsements": 49, "duration_months": 60}, {"name": "Pinecone", "proficiency": "advanced", "endorsements": 31, "duration_months": 54}, {"name": "Information Retrieval", "proficiency": "advanced", "endorsements": 49, "duration_months": 27}, {"name": "Milvus", "proficiency": "expert", "endorsements": 18, "duration_months": 40}, {"name": "QLoRA", "proficiency": "expert", "endorsements": 34, "duration_months": 43}, {"name": "RAG", "proficiency": "expert", "endorsements": 1, "duration_months": 94}, {"name": "Embeddings", "proficiency": "expert", "endorsements": 48, "duration_months": 65}, {"name": "Learning to Rank", "proficiency": "expert", "endorsements": 50, "duration_months": 70}, {"name": "CNN", "proficiency": "intermediate", "endorsements": 9, "duration_months": 16}, {"name": "Go", "proficiency": "beginner", "endorsements": 1, "duration_months": 16}, {"name": "BM25", "proficiency": "advanced", "endorsements": 27, "duration_months": 18}, {"name": "LangChain", "proficiency": "advanced", "endorsements": 15, "duration_months": 53}, {"name": "Weights & Biases", "proficiency": "intermediate", "endorsements": 12, "duration_months": 19}], "certifications": [{"name": "Google Cloud Professional ML Engineer", "issuer": "Google Cloud", "year": 2019}, {"name": "NLP Specialization", "issuer": "Coursera/DeepLearning.AI", "year": 2018}], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 98.0, "signup_date": "2025-03-11", "last_active_date": "2026-05-13", "open_to_work_flag": true, "profile_views_received_30d": 75, "applications_submitted_30d": 18, "recruiter_response_rate": 0.61, "avg_response_time_hours": 59.6, "skill_assessment_scores": {"Deep Learning": 94.0, "Weaviate": 72.4}, "connection_count": 1839, "endorsements_received": 189, "notice_period_days": 15, "expected_salary_range_inr_lpa": {"min": 36.8, "max": 56.6}, "preferred_work_mode": "remote", "willing_to_relocate": true, "github_activity_score": 94.8, "search_appearance_30d": 1291, "saved_by_recruiters_30d": 16, "interview_completion_rate": 0.8, "offer_acceptance_rate": 0.75, "verified_email": true, "verified_phone": true, "linkedin_connected": true}}, {"candidate_id": "CAND_0019480", "profile": {"anonymized_name": "Priya Bhatia", "headline": "NLP Engineer | Search, Ranking & Retrieval", "summary": "Machine learning engineer with 7.4 years of experience building ML-powered features in production. Strong background in NLP, recommendation systems, and applied AI; comfortable across the ML stack from feature engineering through deployment. Recently, I led the team that migrated our keyword-search-based product to embedding-based retrieval. I care a lot about evaluation rigor \\u2014 too many teams ship models without offline benchmarks they trust. My academic background is in CS/ML but my main learning has come from shipping real systems and seeing what holds up under production load. Open to senior IC roles in applied ML or AI engineering, ideally at product companies where I\'d own a meaningful piece of the ML stack.", "location": "Chennai, Tamil Nadu", "country": "India", "years_of_experience": 2.8, "current_title": "NLP Engineer", "current_company": "Meesho", "current_company_size": "1001-5000", "current_industry": "E-commerce"}, "career_history": [{"company": "Meesho", "title": "NLP Engineer", "start_date": "2024-11-03", "end_date": null, "duration_months": 19, "is_current": true, "industry": "E-commerce", "company_size": "1001-5000", "description": "Trained and shipped multiple ranking models for our product\'s discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item engagement history. Owned the offline-online correlation analysis that determined which offline metrics actually predicted A/B test outcomes. Worked closely with PMs to define the optimization target (click-through vs. dwell time vs. downstream conversion) \\u2014 that work was as important as the modeling itself."}, {"company": "InMobi", "title": "Senior Data Scientist", "start_date": "2022-09-15", "end_date": "2024-11-03", "duration_months": 26, "is_current": false, "industry": "AdTech", "company_size": "1001-5000", "description": "Implemented a RAG-based customer support chatbot integrated with our existing ticketing system. Built the document ingestion pipeline (chunking, embedding via OpenAI embeddings, storing in Pinecone) and the answer-generation layer (initially GPT-4, then a fine-tuned smaller model for cost control). Designed the evaluation framework with both automatic metrics (BLEU, ROUGE) and human-in-the-loop quality scores. Deployment cut average ticket resolution time by 31% for the supported categories."}, {"company": "Vedantu", "title": "Senior Data Scientist", "start_date": "2020-04-28", "end_date": "2022-07-17", "duration_months": 27, "is_current": false, "industry": "EdTech", "company_size": "1001-5000", "description": "Trained and shipped multiple ranking models for our product\'s discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item engagement history. Owned the offline-online correlation analysis that determined which offline metrics actually predicted A/B test outcomes. Worked closely with PMs to define the optimization target (click-through vs. dwell time vs. downstream conversion) \\u2014 that work was as important as the modeling itself."}, {"company": "Freshworks", "title": "Search Engineer", "start_date": "2019-02-03", "end_date": "2020-04-28", "duration_months": 15, "is_current": false, "industry": "SaaS", "company_size": "5001-10000", "description": "Trained and shipped multiple ranking models for our product\'s discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item engagement history. Owned the offline-online correlation analysis that determined which offline metrics actually predicted A/B test outcomes. Worked closely with PMs to define the optimization target (click-through vs. dwell time vs. downstream conversion) \\u2014 that work was as important as the modeling itself."}], "education": [{"institution": "Thapar University", "degree": "B.Tech", "field_of_study": "Machine Learning", "start_year": 2004, "end_year": 2007, "grade": "8.11 CGPA", "tier": "tier_2"}], "skills": [{"name": "Terraform", "proficiency": "beginner", "endorsements": 2, "duration_months": 14}, {"name": "LLMs", "proficiency": "expert", "endorsements": 32, "duration_months": 82}, {"name": "Forecasting", "proficiency": "intermediate", "endorsements": 8, "duration_months": 20}, {"name": "TensorFlow", "proficiency": "expert", "endorsements": 47, "duration_months": 55}, {"name": "TTS", "proficiency": "intermediate", "endorsements": 7, "duration_months": 12}, {"name": "Weights & Biases", "proficiency": "intermediate", "endorsements": 6, "duration_months": 29}, {"name": "GANs", "proficiency": "intermediate", "endorsements": 9, "duration_months": 34}, {"name": "Feature Engineering", "proficiency": "intermediate", "endorsements": 6, "duration_months": 28}, {"name": "Milvus", "proficiency": "expert", "endorsements": 31, "duration_months": 58}, {"name": "LangChain", "proficiency": "advanced", "endorsements": 8, "duration_months": 29}, {"name": "Image Classification", "proficiency": "intermediate", "endorsements": 4, "duration_months": 20}, {"name": "BM25", "proficiency": "advanced", "endorsements": 13, "duration_months": 26}, {"name": "OpenSearch", "proficiency": "expert", "endorsements": 5, "duration_months": 41}, {"name": "Prompt Engineering", "proficiency": "advanced", "endorsements": 8, "duration_months": 53}], "certifications": [], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "native"}], "redrob_signals": {"profile_completeness_score": 78.6, "signup_date": "2026-01-10", "last_active_date": "2026-05-13", "open_to_work_flag": true, "profile_views_received_30d": 72, "applications_submitted_30d": 10, "recruiter_response_rate": 0.87, "avg_response_time_hours": 18.7, "skill_assessment_scores": {"LLMs": 75.5, "TensorFlow": 59.8}, "connection_count": 1364, "endorsements_received": 125, "notice_period_days": 90, "expected_salary_range_inr_lpa": {"min": 38.2, "max": 46.1}, "preferred_work_mode": "remote", "willing_to_relocate": true, "github_activity_score": 69.8, "search_appearance_30d": 802, "saved_by_recruiters_30d": 59, "interview_completion_rate": 0.84, "offer_acceptance_rate": -1, "verified_email": true, "verified_phone": true, "linkedin_connected": true}}, {"candidate_id": "CAND_0037000", "profile": {"anonymized_name": "Nikhil Mittal", "headline": "Search Engineer | ML, NLP, Recommendation Systems", "summary": "Machine learning engineer with 6.3 years of experience building ML-powered features in production. Strong background in NLP, recommendation systems, and applied AI; comfortable across the ML stack from feature engineering through deployment. Recently, I led the team that migrated our keyword-search-based product to embedding-based retrieval. I care a lot about evaluation rigor \\u2014 too many teams ship models without offline benchmarks they trust. My academic background is in CS/ML but my main learning has come from shipping real systems and seeing what holds up under production load. Open to senior IC roles in applied ML or AI engineering, ideally at product companies where I\'d own a meaningful piece of the ML stack.", "location": "Noida, Uttar Pradesh", "country": "India", "years_of_experience": 2.7, "current_title": "Search Engineer", "current_company": "Unacademy", "current_company_size": "5001-10000", "current_industry": "EdTech"}, "career_history": [{"company": "Unacademy", "title": "Search Engineer", "start_date": "2024-11-03", "end_date": null, "duration_months": 19, "is_current": true, "industry": "EdTech", "company_size": "5001-10000", "description": "Implemented a RAG-based customer support chatbot integrated with our existing ticketing system. Built the document ingestion pipeline (chunking, embedding via OpenAI embeddings, storing in Pinecone) and the answer-generation layer (initially GPT-4, then a fine-tuned smaller model for cost control). Designed the evaluation framework with both automatic metrics (BLEU, ROUGE) and human-in-the-loop quality scores. Deployment cut average ticket resolution time by 31% for the supported categories."}, {"company": "CRED", "title": "Machine Learning Engineer", "start_date": "2020-11-17", "end_date": "2024-10-27", "duration_months": 48, "is_current": false, "industry": "Fintech", "company_size": "1001-5000", "description": "Trained and shipped multiple ranking models for our product\'s discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item engagement history. Owned the offline-online correlation analysis that determined which offline metrics actually predicted A/B test outcomes. Worked closely with PMs to define the optimization target (click-through vs. dwell time vs. downstream conversion) \\u2014 that work was as important as the modeling itself."}, {"company": "upGrad", "title": "Search Engineer", "start_date": "2020-03-22", "end_date": "2020-11-17", "duration_months": 8, "is_current": false, "industry": "EdTech", "company_size": "1001-5000", "description": "Owned the ranking layer for an e-commerce search product, evolving it from a hand-tuned scoring function to a learning-to-rank model over 9 months. Designed the relevance labeling pipeline (mix of click-through data and explicit human judgments), the feature pipeline, and the training/eval workflow. Most of the work was infrastructure and data quality \\u2014 the modeling part was almost the easy bit. Final model improved revenue-per-search by 12%."}], "education": [{"institution": "SRM Chennai", "degree": "Ph.D", "field_of_study": "Information Technology", "start_year": 2003, "end_year": 2006, "grade": "7.93 CGPA", "tier": "tier_3"}], "skills": [{"name": "Embeddings", "proficiency": "advanced", "endorsements": 15, "duration_months": 39}, {"name": "LangChain", "proficiency": "expert", "endorsements": 9, "duration_months": 87}, {"name": "TTS", "proficiency": "advanced", "endorsements": 20, "duration_months": 51}, {"name": "Semantic Search", "proficiency": "advanced", "endorsements": 24, "duration_months": 39}, {"name": "TensorFlow", "proficiency": "advanced", "endorsements": 43, "duration_months": 31}, {"name": "MLflow", "proficiency": "intermediate", "endorsements": 10, "duration_months": 21}, {"name": "GANs", "proficiency": "intermediate", "endorsements": 12, "duration_months": 36}, {"name": "Hugging Face Transformers", "proficiency": "advanced", "endorsements": 42, "duration_months": 47}, {"name": "Speech Recognition", "proficiency": "advanced", "endorsements": 39, "duration_months": 45}, {"name": "Reinforcement Learning", "proficiency": "intermediate", "endorsements": 7, "duration_months": 27}, {"name": "FAISS", "proficiency": "expert", "endorsements": 10, "duration_months": 77}, {"name": "Pinecone", "proficiency": "advanced", "endorsements": 48, "duration_months": 55}, {"name": "MLOps", "proficiency": "advanced", "endorsements": 48, "duration_months": 58}, {"name": "Next.js", "proficiency": "beginner", "endorsements": 9, "duration_months": 11}, {"name": "BM25", "proficiency": "advanced", "endorsements": 49, "duration_months": 43}, {"name": "Marketing", "proficiency": "intermediate", "endorsements": 14, "duration_months": 23}, {"name": "Sentence Transformers", "proficiency": "advanced", "endorsements": 42, "duration_months": 43}], "certifications": [{"name": "AWS Certified Machine Learning Specialty", "issuer": "AWS", "year": 2023}, {"name": "Google Cloud Professional ML Engineer", "issuer": "Google Cloud", "year": 2021}], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 75.3, "signup_date": "2024-05-07", "last_active_date": "2026-05-16", "open_to_work_flag": false, "profile_views_received_30d": 223, "applications_submitted_30d": 11, "recruiter_response_rate": 0.61, "avg_response_time_hours": 32.2, "skill_assessment_scores": {"Embeddings": 73.1, "LangChain": 51.1, "TTS": 60.7, "Semantic Search": 87.8, "TensorFlow": 72.4}, "connection_count": 705, "endorsements_received": 168, "notice_period_days": 15, "expected_salary_range_inr_lpa": {"min": 26.4, "max": 54.3}, "preferred_work_mode": "remote", "willing_to_relocate": false, "github_activity_score": 66.8, "search_appearance_30d": 942, "saved_by_recruiters_30d": 20, "interview_completion_rate": 0.62, "offer_acceptance_rate": 0.39, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0041669", "profile": {"anonymized_name": "Aisha Banerjee", "headline": "Recommendation Systems Engineer | Search, Ranking & Retrieval", "summary": "Machine learning engineer with 8.0 years of experience building ML-powered features in production. Strong background in NLP, recommendation systems, and applied AI; comfortable across the ML stack from feature engineering through deployment. Recently, I shipped our first RAG-based feature this year and now own the eval framework for it. I\'ve spent enough time debugging production ranking issues to know which signals matter and which are noise. My academic background is in CS/ML but my main learning has come from shipping real systems and seeing what holds up under production load. Open to senior IC roles in applied ML or AI engineering, ideally at product companies where I\'d own a meaningful piece of the ML stack.", "location": "Noida, Uttar Pradesh", "country": "India", "years_of_experience": 8.0, "current_title": "Recommendation Systems Engineer", "current_company": "CRED", "current_company_size": "1001-5000", "current_industry": "Fintech"}, "career_history": [{"company": "CRED", "title": "Recommendation Systems Engineer", "start_date": "2023-05-13", "end_date": null, "duration_months": 37, "is_current": true, "industry": "Fintech", "company_size": "1001-5000", "description": "Owned the ranking layer for an e-commerce search product, evolving it from a hand-tuned scoring function to a learning-to-rank model over 9 months. Designed the relevance labeling pipeline (mix of click-through data and explicit human judgments), the feature pipeline, and the training/eval workflow. Most of the work was infrastructure and data quality \\u2014 the modeling part was almost the easy bit. Final model improved revenue-per-search by 12%."}, {"company": "Mad Street Den", "title": "Search Engineer", "start_date": "2019-02-03", "end_date": "2023-05-13", "duration_months": 52, "is_current": false, "industry": "AI/ML", "company_size": "201-500", "description": "Implemented a RAG-based customer support chatbot integrated with our existing ticketing system. Built the document ingestion pipeline (chunking, embedding via OpenAI embeddings, storing in Pinecone) and the answer-generation layer (initially GPT-4, then a fine-tuned smaller model for cost control). Designed the evaluation framework with both automatic metrics (BLEU, ROUGE) and human-in-the-loop quality scores. Deployment cut average ticket resolution time by 31% for the supported categories."}, {"company": "Yellow.ai", "title": "NLP Engineer", "start_date": "2018-07-31", "end_date": "2019-01-27", "duration_months": 6, "is_current": false, "industry": "AI/ML", "company_size": "201-500", "description": "Built a content recommendation system serving 10M+ users that combined collaborative filtering with content-based ranking. The system uses item-item similarity (via sentence-transformer embeddings) for cold starts and a gradient-boosted model trained on engagement signals for warm users. Most of my time went into the feature pipeline (~200 features) and the A/B testing infrastructure. The launch improved 7-day retention by 6% and time spent per session by 14%."}], "education": [{"institution": "IIT Guwahati", "degree": "B.Tech", "field_of_study": "Information Technology", "start_year": 2006, "end_year": 2010, "grade": "7.04 CGPA", "tier": "tier_1"}, {"institution": "IIT Guwahati", "degree": "B.E.", "field_of_study": "Computer Engineering", "start_year": 2007, "end_year": 2012, "grade": "6.87 CGPA", "tier": "tier_1"}], "skills": [{"name": "FAISS", "proficiency": "advanced", "endorsements": 7, "duration_months": 38}, {"name": "Milvus", "proficiency": "expert", "endorsements": 11, "duration_months": 76}, {"name": "PEFT", "proficiency": "expert", "endorsements": 14, "duration_months": 51}, {"name": "Diffusion Models", "proficiency": "intermediate", "endorsements": 15, "duration_months": 11}, {"name": "MLflow", "proficiency": "advanced", "endorsements": 39, "duration_months": 29}, {"name": "QLoRA", "proficiency": "advanced", "endorsements": 1, "duration_months": 19}, {"name": "Haystack", "proficiency": "advanced", "endorsements": 30, "duration_months": 26}, {"name": "Semantic Search", "proficiency": "expert", "endorsements": 7, "duration_months": 93}, {"name": "Weaviate", "proficiency": "advanced", "endorsements": 59, "duration_months": 18}, {"name": "Prompt Engineering", "proficiency": "advanced", "endorsements": 46, "duration_months": 52}, {"name": "Flask", "proficiency": "intermediate", "endorsements": 10, "duration_months": 20}, {"name": "Weights & Biases", "proficiency": "advanced", "endorsements": 15, "duration_months": 57}, {"name": "ASR", "proficiency": "advanced", "endorsements": 10, "duration_months": 44}, {"name": "MLOps", "proficiency": "advanced", "endorsements": 26, "duration_months": 40}, {"name": "Vue.js", "proficiency": "intermediate", "endorsements": 4, "duration_months": 31}, {"name": "Time Series", "proficiency": "advanced", "endorsements": 54, "duration_months": 37}, {"name": "Information Retrieval", "proficiency": "advanced", "endorsements": 21, "duration_months": 55}, {"name": "TypeScript", "proficiency": "intermediate", "endorsements": 13, "duration_months": 22}, {"name": "Machine Learning", "proficiency": "advanced", "endorsements": 15, "duration_months": 43}, {"name": "Qdrant", "proficiency": "advanced", "endorsements": 37, "duration_months": 18}], "certifications": [], "languages": [{"language": "English", "proficiency": "professional"}, {"language": "Hindi", "proficiency": "conversational"}], "redrob_signals": {"profile_completeness_score": 62.5, "signup_date": "2024-12-20", "last_active_date": "2026-04-06", "open_to_work_flag": true, "profile_views_received_30d": 143, "applications_submitted_30d": 9, "recruiter_response_rate": 0.77, "avg_response_time_hours": 3.7, "skill_assessment_scores": {"FAISS": 83.7, "Milvus": 76.1}, "connection_count": 717, "endorsements_received": 52, "notice_period_days": 60, "expected_salary_range_inr_lpa": {"min": 30.2, "max": 52.8}, "preferred_work_mode": "onsite", "willing_to_relocate": false, "github_activity_score": 70.9, "search_appearance_30d": 326, "saved_by_recruiters_30d": 37, "interview_completion_rate": 0.93, "offer_acceptance_rate": 0.82, "verified_email": true, "verified_phone": true, "linkedin_connected": false}}, {"candidate_id": "CAND_0086022", "profile": {"anonymized_name": "Dhruv Naidu", "headline": "Senior Applied Scientist | LLMs, RAG, Vector Search | ex-Top Tech", "summary": "Senior AI engineer with 5.3 years of hands-on experience building production ML systems, with a focus on search, retrieval, and ranking. Most recently, I led the migration from keyword-based ranking to a learning-to-rank model with embedded behavioral signals, handling peak QPS of 8K with sub-200ms p95. My day-to-day work spans embedding model selection and fine-tuning, hybrid retrieval architecture, learning-to-rank, behavioral-signal integration, and the offline/online evaluation that ties it all together. I\'ve shipped systems in both early-stage product companies and at larger scale, and I\'ve spent enough time on both that I know which tradeoffs apply where. I\'ve made all the standard mistakes \\u2014 embedding everything before defining the metric, over-investing in fine-tuning, optimizing offline metrics that don\'t move online \\u2014 so I notice them faster now. Currently exploring my next move \\u2014 looking for senior IC or tech-lead roles where I can own the intelligence layer end-to-end.", "location": "Kolkata, West Bengal", "country": "India", "years_of_experience": 5.3, "current_title": "Senior Applied Scientist", "current_company": "Sarvam AI", "current_company_size": "51-200", "current_industry": "AI/ML"}, "career_history": [{"company": "Sarvam AI", "title": "Senior Applied Scientist", "start_date": "2024-05-07", "end_date": null, "duration_months": 25, "is_current": true, "industry": "AI/ML", "company_size": "51-200", "description": "Built a RAG-based ranking pipeline serving 50M+ queries per month for an internal recruiter-facing search product. The architecture combined BM25 + dense retrieval (BGE embeddings, FAISS HNSW) with an LLM-based re-ranker on the top-50, falling back to a learning-to-rank model when latency budget was tight. Designed the offline evaluation framework from scratch \\u2014 NDCG, MRR, recall@K calibrated against online A/B engagement metrics. Drove the migration over 4 months including the recruiter-feedback loop that surfaced reranking edge cases."}, {"company": "Uber", "title": "Senior ML Engineer \\u2014 Search & Ranking", "start_date": "2021-02-22", "end_date": "2024-04-07", "duration_months": 38, "is_current": false, "industry": "Transportation", "company_size": "10001+", "description": "Led the migration from keyword-based to embedding-based search across a 30M+ candidate corpus over 8 months. Designed three successive ranker variants and ran them in A/B testing alongside the legacy keyword system. The final embedding ranker improved recruiter engagement metrics by 24% and reduced the average time-to-shortlist by 38%. Most of the engineering effort went into the boring infrastructure: index versioning, embedding versioning, rollback paths, and the dashboards that let recruiters trust the new system. Mentored two junior engineers through this rollout."}], "education": [{"institution": "Stanford University", "degree": "B.Tech", "field_of_study": "Data Science", "start_year": 2016, "end_year": 2020, "grade": "7.52 CGPA", "tier": "tier_1"}], "skills": [{"name": "Vector Search", "proficiency": "advanced", "endorsements": 45, "duration_months": 36}, {"name": "MLflow", "proficiency": "advanced", "endorsements": 37, "duration_months": 46}, {"name": "Recommendation Systems", "proficiency": "advanced", "endorsements": 54, "duration_months": 53}, {"name": "Databricks", "proficiency": "beginner", "endorsements": 6, "duration_months": 15}, {"name": "Deep Learning", "proficiency": "advanced", "endorsements": 0, "duration_months": 28}, {"name": "pgvector", "proficiency": "expert", "endorsements": 53, "duration_months": 78}, {"name": "Fine-tuning LLMs", "proficiency": "expert", "endorsements": 4, "duration_months": 69}, {"name": "Elasticsearch", "proficiency": "advanced", "endorsements": 2, "duration_months": 35}, {"name": "QLoRA", "proficiency": "advanced", "endorsements": 45, "duration_months": 54}, {"name": "Pinecone", "proficiency": "expert", "endorsements": 39, "duration_months": 63}, {"name": "Embeddings", "proficiency": "expert", "endorsements": 12, "duration_months": 71}, {"name": "Kubeflow", "proficiency": "intermediate", "endorsements": 0, "duration_months": 10}, {"name": "PyTorch", "proficiency": "expert", "endorsements": 40, "duration_months": 94}, {"name": "ETL", "proficiency": "intermediate", "endorsements": 13, "duration_months": 15}, {"name": "TensorFlow", "proficiency": "advanced", "endorsements": 16, "duration_months": 20}, {"name": "NLP", "proficiency": "expert", "endorsements": 4, "duration_months": 41}, {"name": "Sentence Transformers", "proficiency": "expert", "endorsements": 32, "duration_months": 80}, {"name": "LoRA", "proficiency": "advanced", "endorsements": 43, "duration_months": 33}, {"name": "LangChain", "proficiency": "expert", "endorsements": 50, "duration_months": 46}], "certifications": [{"name": "LangChain for LLM Application Development", "issuer": "DeepLearning.AI", "year": 2018}, {"name": "AWS Certified Machine Learning Specialty", "issuer": "AWS", "year": 2018}], "languages": [{"language": "English", "proficiency": "native"}, {"language": "Hindi", "proficiency": "professional"}], "redrob_signals": {"profile_completeness_score": 80.1, "signup_date": "2025-07-17", "last_active_date": "2026-04-15", "open_to_work_flag": true, "profile_views_received_30d": 68, "applications_submitted_30d": 6, "recruiter_response_rate": 0.55, "avg_response_time_hours": 59.2, "skill_assessment_scores": {"Vector Search": 92.7, "MLflow": 78.6, "Recommendation Systems": 79.3, "Deep Learning": 79.9, "pgvector": 88.6}, "connection_count": 221, "endorsements_received": 35, "notice_period_days": 0, "expected_salary_range_inr_lpa": {"min": 31.2, "max": 72.6}, "preferred_work_mode": "hybrid", "willing_to_relocate": true, "github_activity_score": 75.2, "search_appearance_30d": 723, "saved_by_recruiters_30d": 53, "interview_completion_rate": 0.68, "offer_acceptance_rate": 0.53, "verified_email": true, "verified_phone": true, "linkedin_connected": true}}]'


## 3. Rank the sample


In [ ]:
import json, pandas as pd
SAMPLE = json.loads(SAMPLE_JSON)
rows = run_ranker(SAMPLE, top_n=20)
df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 170)
print('Ranked %d candidates; top %d:' % (len(SAMPLE), len(rows)))
df


## 4. (Optional) Rank your own sample

Upload a .json (list) or .jsonl (one
record per line) with up to 100 candidates in the candidates.jsonl schema.


In [ ]:
try:
    from google.colab import files
    up = files.upload(); path = next(iter(up)); raw = up[path].decode('utf-8')
except Exception:
    print('Not on Colab / no file - using built-in SAMPLE.'); raw = None

import json, csv, pandas as pd
if raw:
    raw = raw.strip()
    cands = json.loads(raw) if raw[0]=='[' else [json.loads(l) for l in raw.splitlines() if l.strip()]
else:
    cands = json.loads(SAMPLE_JSON)
rows = run_ranker(cands, top_n=min(100, len(cands)))
with open('submission_sample.csv','w',newline='',encoding='utf-8') as fh:
    w=csv.DictWriter(fh, fieldnames=['candidate_id','rank','score','reasoning']); w.writeheader(); w.writerows(rows)
print('Wrote submission_sample.csv with', len(rows), 'rows')
pd.DataFrame(rows)
